In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into DataFrames (stored in kernel for later steps)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Timezone setup (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Helper to add tz-aware datetime column 'ts' from 'timestamp' (seconds)
def add_ts_column(df):
    if 'timestamp' in df.columns:
        # create tz-aware datetime in Asia/Shanghai
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['ts'] = pd.NaT
    return df

metric_df = add_ts_column(metric_df)
trace_df = add_ts_column(trace_df)
log_df = add_ts_column(log_df)
error_logs_df = add_ts_column(error_logs_df)

# Helper to build compact summary for a dataframe
def compact_summary(df, name_col=None, feature_col=None, cmdb_limit=20, feature_limit=50, preview_n=5):
    total_rows = int(df.shape[0])
    # unique cmdb_id values (top N)
    if 'cmdb_id' in df.columns:
        cmdb_vals = pd.Series(df['cmdb_id'].dropna().unique()).astype(str)
        cmdb_vals = sorted(cmdb_vals.tolist())[:cmdb_limit]
    else:
        cmdb_vals = []
    # unique feature/KPI names when provided
    if feature_col and feature_col in df.columns:
        feat_vals = pd.Series(df[feature_col].dropna().unique()).astype(str)
        feat_vals = sorted(feat_vals.tolist())[:feature_limit]
    else:
        # fallback: list other non-system columns (excluding timestamp, ts, cmdb_id, value)
        fallback_cols = [c for c in df.columns if c not in ('timestamp','ts','cmdb_id','value')]
        feat_vals = fallback_cols[:feature_limit]
    # min/max timestamps (tz-aware) as ISO strings or None
    if 'ts' in df.columns and df['ts'].notna().any():
        min_ts = df['ts'].min().isoformat()
        max_ts = df['ts'].max().isoformat()
    else:
        min_ts = None
        max_ts = None
    # preview: small selected columns if possible
    preview_cols = []
    for col in ('ts','cmdb_id', feature_col or '', 'value', 'message'):
        if col and col in df.columns and col not in preview_cols:
            preview_cols.append(col)
    if not preview_cols:  # fallback to first up to 4 columns
        preview_cols = df.columns.tolist()[:4]
    preview = df[preview_cols].head(preview_n).reset_index(drop=True)
    return {
        'total_rows': total_rows,
        'unique_cmdb_ids_top': cmdb_vals,
        'unique_feature_names_top': feat_vals,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'preview_top_rows': preview
    }

metric_summary = compact_summary(metric_df, feature_col='kpi_name')
trace_summary = compact_summary(trace_df, feature_col='trace_name')
log_summary = compact_summary(log_df, feature_col='log_name')
# For error_logs, treat 'message' as the feature/column of interest
error_logs_summary = compact_summary(error_logs_df, feature_col='message')

# Assemble final compact JSON-like dictionary
summary = {
    'metric.csv': metric_summary,
    'trace.csv': trace_summary,
    'log.csv': log_summary,
    'error_logs.csv': error_logs_summary
}

# Display the compact summary variable (keeps full dataframes stored in kernel)
summary

```
Out[1]:
```
```python
# Summary string based on previously loaded dataframes (metric_df, trace_df, log_df, error_logs_df)
summary_text = (
    "Summary of telemetry files (timestamps shown in UTC+8):\n\n"
    "metric.csv: 29,749 rows. Time range 2021-03-25 11:30 to 2021-03-25 12:00. "
    "Top observed cmdb_ids include IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01, Redis02, Tomcat01 and several ServiceTest entries. "
    "Many KPI names are present (JVM, MySQL, container CPU/memory, etc.).\n\n"
    "trace.csv: 7,071 rows. Time range 2021-03-25 11:30 to 2021-03-25 12:00. "
    "Top observed cmdb_ids include IG01, IG02, MG01, MG02, Tomcat01-04 and several docker nodes. "
    "Trace features include trace.self.*, trace.from_*, and trace.to_* metrics (latency p95/mean and row_count).\n\n"
    "log.csv: 502 rows. Time range 2021-03-25 11:30 to 2021-03-25 12:00. "
    "Observed cmdb_ids include Mysql01/02, Tomcat01-04, apache01/02. "
    "Log features are mainly log.error_count, log.gc_pause_max, and log.row_count.\n\n"
    "error_logs.csv: 0 rows (no error log entries in this dataset/time range).\n\n"
    "Overall: metric and trace data are dense in the 30-minute window; logs are sparse; there are no raw error log entries. "
    "This provides enough metric and trace coverage to investigate performance and latency issues for the listed components, "
    "but no error log content is available in error_logs.csv."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 29749, 'unique_cmdb_ids_top': ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01'], 'unique_feature_names_top': ['Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_CpuPercent', 'Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemPercent', 'Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemUsage', 'Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_NetworkRxBytes', 'Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_NetworkTxBytes', 'Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_CpuPercent', 'Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_MemLimit', 'Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_MemPercent', 'Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_MemUsage', 'Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_NetworkTxBytes', 'JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'JVM-Operating System_7778_JVM_JVM_CPULoad', 'JVM-Operating System_7779_JVM_JVM_CPULoad', 'JVM-Runtime_7778_JVM_JVM_Uptime', 'JVM-Runtime_7779_JVM_JVM_Uptime', 'JVM-Threads_7778_JVM_ThreadCount_Threads', 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'Mysql-MySQL_3306_Aborted Clients', 'Mysql-MySQL_3306_Aborted Connects', 'Mysql-MySQL_3306_Binlog Cache Disk Use', 'Mysql-MySQL_3306_Binlog Cache Use', 'Mysql-MySQL_3306_Binlog stmt cache disk use', 'Mysql-MySQL_3306_Binlog stmt cache use', 'Mysql-MySQL_3306_Bytes Received', 'Mysql-MySQL_3306_Bytes Sent', 'Mysql-MySQL_3306_Com Delete', 'Mysql-MySQL_3306_Com Delete Multi', 'Mysql-MySQL_3306_Com Insert', 'Mysql-MySQL_3306_Com Insert Select', 'Mysql-MySQL_3306_Com Load', 'Mysql-MySQL_3306_Com Replace', 'Mysql-MySQL_3306_Com Replace Select', 'Mysql-MySQL_3306_Com Select', 'Mysql-MySQL_3306_Com Update', 'Mysql-MySQL_3306_Com Update Multi', 'Mysql-MySQL_3306_Connections', 'Mysql-MySQL_3306_Created Tmp Disk Tables', 'Mysql-MySQL_3306_Created Tmp Files', 'Mysql-MySQL_3306_Created Tmp Tables', 'Mysql-MySQL_3306_CurrentSQLMaxRunningTime', 'Mysql-MySQL_3306_GetConnectedStateOfMysqld', 'Mysql-MySQL_3306_GetResponseTimeOfMysqld', 'Mysql-MySQL_3306_Handler Commit'], 'min_timestamp': '2021-03-25T11:30:00+08:00', 'max_timestamp': '2021-03-25T12:00:00+08:00', 'preview_top_rows':                          ts cmdb_id                                     kpi_name         value
0 2021-03-25 11:30:00+08:00    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1 2021-03-25 11:30:00+08:00    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.420850e+01
2 2021-03-25 11:30:00+08:00    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  1.257475e+08
3 2021-03-25 11:30:00+08:00    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.846119e+07
4 2021-03-25 11:30:00+08:00    IG01              JVM-Runtime_7778_JVM_JVM_Uptime  1.722045e+06}, 'trace.csv': {'total_rows': 7071, 'unique_cmdb_ids_top': ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 'unique_feature_names_top': ['trace.from_IG01.duration_mean', 'trace.from_IG01.duration_p95', 'trace.from_IG01.row_count', 'trace.from_IG02.duration_mean', 'trace.from_IG02.duration_p95', 'trace.from_IG02.row_count', 'trace.from_MG01.duration_mean', 'trace.from_MG01.duration_p95', 'trace.from_MG01.row_count', 'trace.from_MG02.duration_mean', 'trace.from_MG02.duration_p95', 'trace.from_MG02.row_count', 'trace.from_Tomcat01.duration_mean', 'trace.from_Tomcat01.duration_p95', 'trace.from_Tomcat01.row_count', 'trace.from_Tomcat02.duration_mean', 'trace.from_Tomcat02.duration_p95', 'trace.from_Tomcat02.row_count', 'trace.from_Tomcat03.duration_mean', 'trace.from_Tomcat03.duration_p95', 'trace.from_Tomcat03.row_count', 'trace.from_Tomcat04.duration_mean', 'trace.from_Tomcat04.duration_p95', 'trace.from_Tomcat04.row_count', 'trace.from_dockerA1.duration_mean', 'trace.from_dockerA1.duration_p95', 'trace.from_dockerA1.row_count', 'trace.from_dockerA2.duration_mean', 'trace.from_dockerA2.duration_p95', 'trace.from_dockerA2.row_count', 'trace.from_dockerB1.duration_mean', 'trace.from_dockerB1.duration_p95', 'trace.from_dockerB1.row_count', 'trace.from_dockerB2.duration_mean', 'trace.from_dockerB2.duration_p95', 'trace.from_dockerB2.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count', 'trace.to_Tomcat01.duration_mean', 'trace.to_Tomcat01.duration_p95'], 'min_timestamp': '2021-03-25T11:30:00+08:00', 'max_timestamp': '2021-03-25T12:00:00+08:00', 'preview_top_rows':                          ts cmdb_id                       trace_name        value
0 2021-03-25 11:30:00+08:00    IG01         trace.self.duration_mean     0.538697
1 2021-03-25 11:30:00+08:00    IG01          trace.self.duration_p95     2.345200
2 2021-03-25 11:30:00+08:00    IG01             trace.self.row_count  1794.000000
3 2021-03-25 11:30:00+08:00    IG01  trace.to_Tomcat01.duration_mean     0.471446
4 2021-03-25 11:30:00+08:00    IG01   trace.to_Tomcat01.duration_p95     2.116850}, 'log.csv': {'total_rows': 502, 'unique_cmdb_ids_top': ['Mysql01', 'Mysql02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02'], 'unique_feature_names_top': ['log.error_count', 'log.gc_pause_max', 'log.row_count'], 'min_timestamp': '2021-03-25T11:30:00+08:00', 'max_timestamp': '2021-03-25T12:00:00+08:00', 'preview_top_rows':                          ts   cmdb_id         log_name  value
0 2021-03-25 11:30:00+08:00  Tomcat01  log.error_count    0.0
1 2021-03-25 11:30:00+08:00  Tomcat01    log.row_count  454.0
2 2021-03-25 11:30:00+08:00  Tomcat02  log.error_count    0.0
3 2021-03-25 11:30:00+08:00  Tomcat02    log.row_count  437.0
4 2021-03-25 11:30:00+08:00  Tomcat03  log.error_count    0.0}, 'error_logs.csv': {'total_rows': 0, 'unique_cmdb_ids_top': [], 'unique_feature_names_top': [], 'min_timestamp': None, 'max_timestamp': None, 'preview_top_rows': Empty DataFrame
Columns: [ts, cmdb_id, message]
Index: []}}```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse metric_df and tz from previous kernel state
# Ensure tz is set
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric_df
thresholds_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95_threshold'})
)

# 2) Filter metric_df to incident time window inclusive and save as metric_incident_df
start_ts = pd.Timestamp('2021-03-25 11:30:00').tz_localize(tz)
end_ts = pd.Timestamp('2021-03-25 12:00:00').tz_localize(tz)

# metric_df already has 'ts' tz-aware column
metric_incident_df = metric_df[(metric_df['ts'] >= start_ts) & (metric_df['ts'] <= end_ts)].copy()

# 3) For each pair present in metric_incident_df compute anomaly_count and other metrics

# max value in window per series
max_in_window = (
    metric_incident_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .max()
    .rename(columns={'value': 'max_value_in_window'})
)

# join thresholds to incident data to evaluate anomalies per row
merged_incident = metric_incident_df.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')

# identify anomaly rows where value >= p95_threshold (handle missing thresholds as no anomaly)
anomaly_rows = merged_incident[merged_incident['p95_threshold'].notna() & (merged_incident['value'] >= merged_incident['p95_threshold'])].copy()

# aggregate anomaly info: count and earliest timestamp
if not anomaly_rows.empty:
    anomaly_agg = (
        anomaly_rows
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_anomaly_ts=('ts', 'min')
        )
    )
else:
    anomaly_agg = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_ts'])

# Combine aggregates: ensure every series in the incident window is present
summary_df = (
    max_in_window
    .merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NaNs for series with zero anomalies
summary_df['anomaly_count'] = summary_df['anomaly_count'].fillna(0).astype(int)
summary_df['earliest_anomaly_ts'] = pd.to_datetime(summary_df['earliest_anomaly_ts']).dt.tz_localize(None)
# Reattach tz info (if not null) for display consistency
# If earliest_anomaly_ts is NaT, keep as NaT; else localize to tz
summary_df['earliest_anomaly_ts'] = summary_df['earliest_anomaly_ts'].apply(
    lambda x: (pd.Timestamp(x).tz_localize(tz) if not pd.isna(x) else pd.NaT)
)

# compute ratio, guard division by zero or missing threshold
def safe_ratio(row):
    th = row['p95_threshold']
    mv = row['max_value_in_window']
    if pd.isna(th) or th == 0:
        return np.nan
    return mv / th

summary_df['max_over_threshold_ratio'] = summary_df.apply(safe_ratio, axis=1)

# Sort by anomaly_count desc, then ratio desc, and select requested columns
result_df = (
    summary_df
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_ts', 'p95_threshold', 'max_value_in_window', 'max_over_threshold_ratio']]
    .sort_values(by=['anomaly_count', 'max_over_threshold_ratio'], ascending=[False, False])
    .reset_index(drop=True)
)

# Convert earliest_anomaly_ts to ISO strings for compact display
result_df['earliest_anomaly_ts'] = result_df['earliest_anomaly_ts'].apply(
    lambda x: x.isoformat() if not pd.isna(x) else None
)

# total number of distinct (cmdb_id, kpi_name) series scanned in incident window
total_series_scanned = result_df.shape[0]

# Top 20 rows as compact table
top20_df = result_df.head(20)

# Keep thresholds_df and metric_incident_df in kernel for later use (they are already variables)
top20_df, total_series_scanned, thresholds_df.shape, metric_incident_df.shape

```
Out[2]:
```
```python
summary_text = (
    "Summary of the anomaly scan (all timestamps in UTC+8):\n\n"
    "1) thresholds_df: computed global P95 per (cmdb_id, kpi_name). "
    "There are 1,754 distinct (cmdb_id, kpi_name) series with P95 thresholds.\n\n"
    "2) metric_incident_df: metric rows in the incident window 2021-03-25 11:30:00+08:00 "
    "through 2021-03-25 12:00:00+08:00 contain 29,749 rows (stored in kernel).\n\n"
    "3) Anomaly aggregation results (top findings):\n"
    "- We scanned 1,754 series present in the incident window. The top series by anomaly_count:\n"
    "  * MG01 | OSLinux-..._zabbix-zabbix_agentd_... : anomaly_count=31, earliest=2021-03-25T11:30:00+08:00, "
    "p95=0.30, max=0.40, max_over_threshold_ratio≈1.33\n"
    "  * IG01 | JVM-Memory_7778_JVM_Memory_HeapMemoryMax : anomaly_count=31, earliest=2021-03-25T11:30:00+08:00, "
    "p95=2.040136e+09, max=2.040136e+09, ratio=1.00\n"
    "  * IG01 | JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed : anomaly_count=31, earliest=2021-03-25T11:30:00+08:00, "
    "p95≈5.8461e+07, max≈5.8461e+07, ratio=1.00\n"
    "  * IG02 / MG01 / MG02 and several ServiceTest series also appear in the top-20, "
    "many with anomaly_count=31 (anomalous at every minute in the 31-minute window).\n\n"
    "4) Interpretation / next steps: many series hit or exceeded their global P95 for every sample in the window (31 points), "
    "indicating sustained values at or above the P95 for those KPIs. The largest ratio in the top-20 is ~1.33 (MG01 zabbix agent). "
    "Most JVM memory-related metrics hit their P95 exactly (ratio ~1.00).\n\n"
    "Stored variables in kernel for follow-up: thresholds_df (P95 per series) and metric_incident_df (incident-window rows)."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                           kpi_name  anomaly_count        earliest_anomaly_ts  p95_threshold  max_value_in_window  max_over_threshold_ratio
0            MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-25T11:30:00+08:00   3.000000e-01         4.000000e-01                  1.333333
1            IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-25T11:30:00+08:00   2.040136e+09         2.040136e+09                  1.000000
2            IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31  2021-03-25T11:30:00+08:00   5.846119e+07         5.846119e+07                  1.000000
3            IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-25T11:30:00+08:00   2.040136e+09         2.040136e+09                  1.000000
4            IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31  2021-03-25T11:30:00+08:00   5.860966e+07         5.860966e+07                  1.000000
5            MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31  2021-03-25T11:30:00+08:00   2.040136e+09         2.040136e+09                  1.000000
6            MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31  2021-03-25T11:30:00+08:00   5.881820e+07         5.881820e+07                  1.000000
7            MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31  2021-03-25T11:30:00+08:00   2.000000e+00         2.000000e+00                  1.000000
8            MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31  2021-03-25T11:30:00+08:00   3.000000e+00         3.000000e+00                  1.000000
9            MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31  2021-03-25T11:30:00+08:00   2.040136e+09         2.040136e+09                  1.000000
10           MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31  2021-03-25T11:30:00+08:00   5.899703e+07         5.899703e+07                  1.000000
11           MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-25T11:30:00+08:00   3.000000e-01         3.000000e-01                  1.000000
12   ServiceTest1                                             app.rr             31  2021-03-25T11:30:00+08:00   1.000000e+02         1.000000e+02                  1.000000
13   ServiceTest1                                             app.sr             31  2021-03-25T11:30:00+08:00   1.000000e+02         1.000000e+02                  1.000000
14  ServiceTest10                                             app.rr             31  2021-03-25T11:30:00+08:00   1.000000e+02         1.000000e+02                  1.000000
15  ServiceTest10                                             app.sr             31  2021-03-25T11:30:00+08:00   1.000000e+02         1.000000e+02                  1.000000
16  ServiceTest11                                             app.rr             31  2021-03-25T11:30:00+08:00   1.000000e+02         1.000000e+02                  1.000000
17  ServiceTest11                                             app.sr             31  2021-03-25T11:30:00+08:00   1.000000e+02         1.000000e+02                  1.000000
18   ServiceTest3                                             app.rr             31  2021-03-25T11:30:00+08:00   1.000000e+02         1.000000e+02                  1.000000
19   ServiceTest3                                             app.sr             31  2021-03-25T11:30:00+08:00   1.000000e+02         1.000000e+02                  1.000000, 1754, (1754, 3), (29749, 5))```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Helper to ensure tz-aware timestamp in Asia/Shanghai or NaT
def to_tz(x):
    if pd.isna(x):
        return pd.NaT
    ts = pd.to_datetime(x)
    if ts.tzinfo is None:
        return ts.tz_localize(tz)
    return ts.tz_convert(tz)

# Helper to compute safe ratio
def safe_ratio(th, mv):
    if pd.isna(th) or th == 0:
        return np.nan
    return mv / th

# Recompute per-series anomaly summary from metric_incident_df and thresholds_df
merged = metric_incident_df.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')

# Identify anomaly rows where value >= p95_threshold (and threshold exists)
anomaly_rows = merged[merged['p95_threshold'].notna() & (merged['value'] >= merged['p95_threshold'])].copy()

# Per-series max value in the incident window
max_in_window = (
    metric_incident_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .max()
    .rename(columns={'value': 'max_value_in_window'})
)

# Aggregate anomaly rows per series
if not anomaly_rows.empty:
    anomaly_agg = (
        anomaly_rows
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_anomaly_ts=('ts', 'min')
        )
    )
else:
    anomaly_agg = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_ts'])

# Build per-series DataFrame
per_series = (
    max_in_window
    .merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id', 'kpi_name'], how='left')
)

per_series['anomaly_count'] = per_series['anomaly_count'].fillna(0).astype(int)
per_series['earliest_anomaly_ts'] = per_series['earliest_anomaly_ts'].apply(lambda x: to_tz(x) if not pd.isna(x) else pd.NaT)
per_series['max_over_threshold_ratio'] = per_series.apply(lambda r: safe_ratio(r['p95_threshold'], r['max_value_in_window']), axis=1)

# Aggregate by component (cmdb_id)
# num_kpi_anomalies: distinct kpi_name with anomaly_count > 0
num_kpi_anomalies = (
    per_series[per_series['anomaly_count'] > 0]
    .groupby('cmdb_id', as_index=False)['kpi_name']
    .nunique()
    .rename(columns={'kpi_name': 'num_kpi_anomalies'})
)

# total_anomaly_points: sum of anomaly_count per component
total_anomaly_points = (
    per_series.groupby('cmdb_id', as_index=False)['anomaly_count']
    .sum()
    .rename(columns={'anomaly_count': 'total_anomaly_points'})
)

# earliest_anomaly_ts per component
earliest_per_comp = (
    per_series[per_series['anomaly_count'] > 0]
    .groupby('cmdb_id', as_index=False)['earliest_anomaly_ts']
    .min()
)

# max_over_threshold_ratio per component
max_ratio_per_comp = (
    per_series.groupby('cmdb_id', as_index=False)['max_over_threshold_ratio']
    .max()
    .rename(columns={'max_over_threshold_ratio': 'max_over_threshold_ratio'})
)

# top_kpi per component: highest ratio, tie-breaker by anomaly_count desc, then kpi_name lexicographically
rank_df = per_series[['cmdb_id', 'kpi_name', 'anomaly_count', 'max_over_threshold_ratio']].copy()
rank_df['ratio_for_rank'] = rank_df['max_over_threshold_ratio'].fillna(-np.inf)
rank_df = rank_df.sort_values(['cmdb_id', 'ratio_for_rank', 'anomaly_count', 'kpi_name'], ascending=[True, False, False, True])
top_kpi_per_comp = rank_df.groupby('cmdb_id', as_index=False).first()[['cmdb_id', 'kpi_name']].rename(columns={'kpi_name': 'top_kpi'})

# Compose final component-level DataFrame
component_anomaly_df = (
    total_anomaly_points
    .merge(num_kpi_anomalies, on='cmdb_id', how='left')
    .merge(earliest_per_comp, on='cmdb_id', how='left')
    .merge(max_ratio_per_comp, on='cmdb_id', how='left')
    .merge(top_kpi_per_comp, on='cmdb_id', how='left')
)

component_anomaly_df['num_kpi_anomalies'] = component_anomaly_df['num_kpi_anomalies'].fillna(0).astype(int)
component_anomaly_df['earliest_anomaly_ts'] = component_anomaly_df['earliest_anomaly_ts'].apply(lambda x: to_tz(x) if not pd.isna(x) else pd.NaT)
component_anomaly_df['top_kpi'] = component_anomaly_df['top_kpi'].where(component_anomaly_df['top_kpi'].notna(), None)

# Sort components by total_anomaly_points desc, then max_over_threshold_ratio desc
component_anomaly_df = component_anomaly_df.sort_values(by=['total_anomaly_points', 'max_over_threshold_ratio'], ascending=[False, False]).reset_index(drop=True)

# Prepare compact top-20 display (convert earliest_anomaly_ts to ISO strings)
display_df = component_anomaly_df.copy()
display_df['earliest_anomaly_ts'] = display_df['earliest_anomaly_ts'].apply(lambda x: x.isoformat() if not pd.isna(x) else None)

top20_components = display_df.loc[:, ['cmdb_id', 'num_kpi_anomalies', 'total_anomaly_points', 'earliest_anomaly_ts', 'max_over_threshold_ratio', 'top_kpi']].head(20)

# Total number of components scanned (those appearing in the incident window)
total_components_scanned = component_anomaly_df.shape[0]

# Store component_anomaly_df in kernel (variable exists now) and return compact results
top20_components, total_components_scanned, component_anomaly_df.shape

```
Out[3]:
```
```python
summary_text = (
    "Component-level anomaly summary (incident window 2021-03-25 11:30–12:00 +08:00):\n\n"
    "Total components scanned: 26 (component_anomaly_df stored in kernel).\n\n"
    "Top components by total anomaly points:\n"
    "1) Mysql01 — 2,652 anomaly points across 226 KPIs; earliest anomaly 2021-03-25T11:30:00+08:00; "
    "max_over_threshold_ratio ≈ 4.51; top_kpi = OSLinux-CPU_CPU_CPUCpuUtil.\n"
    "2) Mysql02 — 2,394 anomaly points across 224 KPIs; earliest 2021-03-25T11:30:00+08:00; "
    "ratio = 1.60; top_kpi = OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps.\n"
    "3) Tomcat02 — 1,298 points across 121 KPIs; earliest 11:30; ratio ≈ 6.72; top_kpi = OSLinux-CPU_CPU_CPUWio.\n"
    "4) Tomcat04 — 1,264 points across 121 KPIs; earliest 11:30; ratio ≈ 3.78; top_kpi = local disk read metric.\n"
    "5) Tomcat03 — 1,255 points across 121 KPIs; earliest 11:30; ratio = 4.00; top_kpi = CPU idle metric.\n"
    "6) Tomcat01 — 1,151 points across 118 KPIs; earliest 11:30; ratio ≈ 2.45; top_kpi = local disk RTps metric.\n"
    "7) Redis02 — 1,097 points across 110 KPIs; earliest 11:30; ratio ≈ 5.43; top_kpi = CPU WIO metric.\n"
    "8) Redis01 — 994 points across 113 KPIs; earliest 11:30; ratio ≈ 3.47; top_kpi = CPU WIO metric.\n"
    "9) MG02 — 790 points across 104 KPIs; earliest 11:30; ratio = 4.00; top_kpi = local disk DSKBps.\n"
    "10) MG01 — 747 points across 105 KPIs; earliest 11:30; ratio ≈ 2.20; top_kpi = CPU WIO metric.\n\n"
    "Notes/interpretation:\n"
    "- Many components show sustained anomalies (values >= their global P95) at every minute in the 31-minute window. "
    "Mysql01/Mysql02 and the Tomcat/Redis fleets are the most affected by total anomaly points.  \n"
    "- High max_over_threshold_ratio (e.g., Tomcat02 ≈6.7, Mysql01 ≈4.5) indicates some KPIs spiked well above their historical P95.  \n"
    "- Earliest anomalies for top components all start at 2021-03-25T11:30:00+08:00, indicating the issue was present at the window start.\n\n"
    "Stored variables for follow-up: component_anomaly_df, thresholds_df, metric_incident_df."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  num_kpi_anomalies  total_anomaly_points        earliest_anomaly_ts  max_over_threshold_ratio                                          top_kpi
0         Mysql01                226                  2652  2021-03-25T11:30:00+08:00                  4.508701                       OSLinux-CPU_CPU_CPUCpuUtil
1         Mysql02                224                  2394  2021-03-25T11:30:00+08:00                  1.600000   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps
2        Tomcat02                121                  1298  2021-03-25T11:30:00+08:00                  6.715785                           OSLinux-CPU_CPU_CPUWio
3        Tomcat04                121                  1264  2021-03-25T11:30:00+08:00                  3.777228  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
4        Tomcat03                121                  1255  2021-03-25T11:30:00+08:00                  4.000000                  OSLinux-CPU_CPU-2_SingleCpuidle
5        Tomcat01                118                  1151  2021-03-25T11:30:00+08:00                  2.452456  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps
6         Redis02                110                  1097  2021-03-25T11:30:00+08:00                  5.429517                           OSLinux-CPU_CPU_CPUWio
7         Redis01                113                   994  2021-03-25T11:30:00+08:00                  3.469232                           OSLinux-CPU_CPU_CPUWio
8            MG02                104                   790  2021-03-25T11:30:00+08:00                  4.000000   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps
9            MG01                105                   747  2021-03-25T11:30:00+08:00                  2.197619                           OSLinux-CPU_CPU_CPUWio
10           IG01                 88                   579  2021-03-25T11:30:00+08:00                  3.646505                           OSLinux-CPU_CPU_CPUWio
11           IG02                 88                   549  2021-03-25T11:30:00+08:00                 10.000000     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
12       apache02                 80                   492  2021-03-25T11:30:00+08:00                  3.127489                           OSLinux-CPU_CPU_CPUWio
13       apache01                 81                   491  2021-03-25T11:30:00+08:00                  1.778017                  OSLinux-CPU_CPU-0_SingleCpuUtil
14  ServiceTest10                  4                    66  2021-03-25T11:30:00+08:00                  6.954892                                          app.mrt
15   ServiceTest5                  4                    66  2021-03-25T11:30:00+08:00                  3.554871                                          app.mrt
16   ServiceTest1                  4                    66  2021-03-25T11:30:00+08:00                  3.043948                                          app.mrt
17   ServiceTest3                  4                    66  2021-03-25T11:30:00+08:00                  2.095659                                          app.mrt
18   ServiceTest8                  4                    66  2021-03-25T11:30:00+08:00                  1.872653                                          app.mrt
19   ServiceTest9                  4                    66  2021-03-25T11:30:00+08:00                  1.708061                                          app.mrt, 26, (26, 6))```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_df and tz if present; ensure tz is set
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name)
p95 = trace_df.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value': 'p95_threshold'})
p5 = trace_df.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.05).rename(columns={'value': 'p5_threshold'})

thresholds_trace_df = p95.merge(p5, on=['cmdb_id', 'trace_name'], how='outer')

# 2) Filter trace_df to incident time window (inclusive) and save
start_ts = pd.Timestamp('2021-03-25 11:30:00').tz_localize(tz)
end_ts = pd.Timestamp('2021-03-25 12:00:00').tz_localize(tz)

trace_incident_df = trace_df[(trace_df['ts'] >= start_ts) & (trace_df['ts'] <= end_ts)].copy()

# 3) Per (cmdb_id, trace_name) in incident window compute required stats
# Merge thresholds into incident data
merged = trace_incident_df.merge(thresholds_trace_df, on=['cmdb_id', 'trace_name'], how='left')

# Flags for high/low (only when corresponding threshold exists)
merged['is_high'] = merged['p95_threshold'].notna() & (merged['value'] >= merged['p95_threshold'])
merged['is_low']  = merged['p5_threshold'].notna()  & (merged['value'] <= merged['p5_threshold'])

# Earliest ts values conditioned
merged['earliest_high_ts_val'] = merged['ts'].where(merged['is_high'], pd.NaT)
merged['earliest_low_ts_val']  = merged['ts'].where(merged['is_low'], pd.NaT)

# Per-series aggregates
per_trace_agg = (
    merged
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count_high=('is_high', 'sum'),
        count_low=('is_low', 'sum'),
        earliest_high_ts=('earliest_high_ts_val', 'min'),
        earliest_low_ts=('earliest_low_ts_val', 'min'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
        p95_threshold=('p95_threshold', 'first'),
        p5_threshold=('p5_threshold', 'first')
    )
)

# Compute ratios with guards for zero / missing thresholds
def safe_div(n, d):
    if pd.isna(n) or pd.isna(d) or d == 0:
        return np.nan
    return n / d

per_trace_agg['max_over_p95'] = per_trace_agg.apply(lambda r: safe_div(r['max_value_in_window'], r['p95_threshold']), axis=1)
per_trace_agg['min_under_p5_ratio'] = per_trace_agg.apply(lambda r: safe_div(r['p5_threshold'], r['min_value_in_window']), axis=1)

# Ensure integer counts
per_trace_agg['count_high'] = per_trace_agg['count_high'].astype(int)
per_trace_agg['count_low'] = per_trace_agg['count_low'].astype(int)

# 4) Aggregate per cmdb_id
# num_trace_kpi_anomalies: distinct trace_name with count_high>0 or count_low>0
anomalous_series = per_trace_agg[(per_trace_agg['count_high'] > 0) | (per_trace_agg['count_low'] > 0)]

num_trace_kpi_anomalies = (
    anomalous_series
    .groupby('cmdb_id', as_index=False)['trace_name']
    .nunique()
    .rename(columns={'trace_name': 'num_trace_kpi_anomalies'})
)

# total_trace_anomaly_points = sum(count_high + count_low)
per_trace_agg['sum_counts'] = per_trace_agg['count_high'] + per_trace_agg['count_low']
total_trace_anomaly_points = (
    per_trace_agg
    .groupby('cmdb_id', as_index=False)['sum_counts']
    .sum()
    .rename(columns={'sum_counts': 'total_trace_anomaly_points'})
)

# earliest_trace_anomaly_ts = earliest among earliest_high_ts and earliest_low_ts per component
per_trace_agg['earliest_any_ts'] = per_trace_agg[['earliest_high_ts', 'earliest_low_ts']].min(axis=1)
earliest_trace_anomaly_ts = (
    per_trace_agg[per_trace_agg['earliest_any_ts'].notna()]
    .groupby('cmdb_id', as_index=False)['earliest_any_ts']
    .min()
    .rename(columns={'earliest_any_ts': 'earliest_trace_anomaly_ts'})
)

# max_trace_over_threshold_ratio = maximum max_over_p95 across its trace KPIs
max_trace_over = (
    per_trace_agg
    .groupby('cmdb_id', as_index=False)['max_over_p95']
    .max()
    .rename(columns={'max_over_p95': 'max_trace_over_threshold_ratio'})
)

# top_trace_kpi: choose trace_name with highest max_over_p95; tie-breaker by count_low desc, then count_high desc, then trace_name
rank_df = per_trace_agg.copy()
rank_df['rank_max_over_p95'] = rank_df['max_over_p95'].fillna(-np.inf)
rank_df = rank_df.sort_values(['cmdb_id', 'rank_max_over_p95', 'count_low', 'count_high', 'trace_name'],
                              ascending=[True, False, False, False, True])
top_trace_kpi = rank_df.groupby('cmdb_id', as_index=False).first()[['cmdb_id', 'trace_name']].rename(columns={'trace_name': 'top_trace_kpi'})

# Compose final component-level aggregation
component_trace_agg = (
    total_trace_anomaly_points
    .merge(num_trace_kpi_anomalies, on='cmdb_id', how='left')
    .merge(earliest_trace_anomaly_ts, on='cmdb_id', how='left')
    .merge(max_trace_over, on='cmdb_id', how='left')
    .merge(top_trace_kpi, on='cmdb_id', how='left')
)

component_trace_agg['num_trace_kpi_anomalies'] = component_trace_agg['num_trace_kpi_anomalies'].fillna(0).astype(int)
component_trace_agg['earliest_trace_anomaly_ts'] = component_trace_agg['earliest_trace_anomaly_ts'].apply(
    lambda x: x.isoformat() if not pd.isna(x) else None
)

# 5) Return compact top-20 components sorted by total_trace_anomaly_points desc, tie-breaker max_trace_over_threshold_ratio desc
component_trace_agg = component_trace_agg.sort_values(['total_trace_anomaly_points', 'max_trace_over_threshold_ratio'],
                                                      ascending=[False, False]).reset_index(drop=True)

top20_trace_components = component_trace_agg.loc[:, ['cmdb_id', 'num_trace_kpi_anomalies', 'total_trace_anomaly_points', 'earliest_trace_anomaly_ts', 'max_trace_over_threshold_ratio', 'top_trace_kpi']].head(20)

total_components_scanned = component_trace_agg.shape[0]

# Keep thresholds_trace_df and trace_incident_df in kernel (they are defined). Display compact results.
top20_trace_components, total_components_scanned, thresholds_trace_df.shape, trace_incident_df.shape

```
Out[4]:
```
```python
summary_text = (
    "Trace anomaly scan summary (timestamps in UTC+8):\n\n"
    "Data stored in kernel: thresholds_trace_df (261 trace series with global P5/P95) and trace_incident_df (7,071 rows in the incident window).\n\n"
    "Components scanned: 12.\n\n"
    "Top components by total trace anomaly points (count_high + count_low):\n"
    "1) MG01 — 42 trace KPIs anomalous, 166 anomaly points; earliest anomaly 2021-03-25T11:30:00+08:00; "
    "max_over_p95 ≈ 2.63; top_trace_kpi = trace.from_dockerB1.duration_mean.\n"
    "2) MG02 — 42 KPIs, 164 points; earliest 11:30; max_over_p95 ≈ 2.71; top_trace_kpi = trace.from_dockerA1.duration_mean.\n"
    "3) dockerA1 — 18 KPIs, 72 points; earliest 11:30; max_over_p95 ≈ 2.71; top_trace_kpi = trace.to_MG02.duration_mean.\n"
    "4) dockerA2 — 18 KPIs, 72 points; earliest 11:30; max_over_p95 ≈ 1.79; top_trace_kpi = trace.to_MG02.duration_mean.\n"
    "5) Tomcat01 — 18 KPIs, 70 points; earliest 11:30; max_over_p95 ≈ 1.84; top_trace_kpi = trace.from_IG02.duration_p95.\n"
    "6) dockerB1 — 18 KPIs, 69 points; earliest 11:30; max_over_p95 ≈ 2.63; top_trace_kpi = trace.to_MG01.duration_mean.\n"
    "7) Tomcat02 — 18 KPIs, 69 points; earliest 11:30; max_over_p95 ≈ 1.43; top_trace_kpi = trace.to_MG01.duration_p95.\n"
    "8) dockerB2 — 18 KPIs, 68 points; earliest 11:30; max_over_p95 ≈ 4.00; top_trace_kpi = trace.self.duration_mean.\n"
    "9) IG02 — 18 KPIs, 67 points; earliest 11:30; max_over_p95 ≈ 1.89; top_trace_kpi = trace.self.duration_p95.\n"
    "10) Tomcat03 — 18 KPIs, 67 points; earliest 11:30; max_over_p95 ≈ 1.77; top_trace_kpi = trace.from_IG02.duration_p95.\n"
    "11) Tomcat04 — 18 KPIs, 66 points; earliest 11:30; max_over_p95 ≈ 1.38; top_trace_kpi = trace.to_MG02.duration_mean.\n"
    "12) IG01 — 15 KPIs, 60 points; earliest anomaly 2021-03-25T11:32:00+08:00; max_over_p95 ≈ 1.78; top_trace_kpi = trace.self.duration_p95.\n\n"
    "Interpretation / next steps:\n"
    "- MG01 and MG02 show the largest number of trace anomalies (both in distinct KPIs and total anomaly points), with anomalies present from the start of the incident window.  \n"
    "- Several docker and Tomcat nodes also show elevated trace anomalies (latency-related trace.* metrics), and dockerB2 shows the largest per-kpi ratio (~4.0) among the top components.  \n"
    "- Most earliest anomalies occur at 11:30:00+08:00, indicating the issue appears at the window start for most components; IG01's traces began showing anomalies slightly later (11:32).  \n"
    "- Use thresholds_trace_df and per-series aggregates in trace_incident_df to drill down into the specific trace metrics (top_trace_kpi) for root-cause analysis (latency spikes, slow downstream calls, etc.)."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  num_trace_kpi_anomalies  total_trace_anomaly_points  earliest_trace_anomaly_ts  max_trace_over_threshold_ratio                      top_trace_kpi
0       MG01                       42                         166  2021-03-25T11:30:00+08:00                        2.630241  trace.from_dockerB1.duration_mean
1       MG02                       42                         164  2021-03-25T11:30:00+08:00                        2.714840  trace.from_dockerA1.duration_mean
2   dockerA1                       18                          72  2021-03-25T11:30:00+08:00                        2.714840        trace.to_MG02.duration_mean
3   dockerA2                       18                          72  2021-03-25T11:30:00+08:00                        1.790020        trace.to_MG02.duration_mean
4   Tomcat01                       18                          70  2021-03-25T11:30:00+08:00                        1.841055       trace.from_IG02.duration_p95
5   dockerB1                       18                          69  2021-03-25T11:30:00+08:00                        2.630241        trace.to_MG01.duration_mean
6   Tomcat02                       18                          69  2021-03-25T11:30:00+08:00                        1.426897         trace.to_MG01.duration_p95
7   dockerB2                       18                          68  2021-03-25T11:30:00+08:00                        3.996830           trace.self.duration_mean
8       IG02                       18                          67  2021-03-25T11:30:00+08:00                        1.886607            trace.self.duration_p95
9   Tomcat03                       18                          67  2021-03-25T11:30:00+08:00                        1.765097       trace.from_IG02.duration_p95
10  Tomcat04                       18                          66  2021-03-25T11:30:00+08:00                        1.377668        trace.to_MG02.duration_mean
11      IG01                       15                          60  2021-03-25T11:32:00+08:00                        1.780702            trace.self.duration_p95, 12, (261, 4), (7071, 5))```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse thresholds_trace_df and trace_incident_df from kernel

# 1) Merge thresholds into incident traces and compute per (cmdb_id, trace_name) aggregates
merged = trace_incident_df.merge(thresholds_trace_df, on=['cmdb_id', 'trace_name'], how='left')

merged['is_high'] = merged['p95_threshold'].notna() & (merged['value'] >= merged['p95_threshold'])
merged['is_low']  = merged['p5_threshold'].notna()  & (merged['value'] <= merged['p5_threshold'])

merged['earliest_high_ts_val'] = merged['ts'].where(merged['is_high'])
merged['earliest_low_ts_val']  = merged['ts'].where(merged['is_low'])

per_trace = (
    merged
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count_high=('is_high', 'sum'),
        count_low=('is_low', 'sum'),
        earliest_high_ts=('earliest_high_ts_val', 'min'),
        earliest_low_ts=('earliest_low_ts_val', 'min'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
        p95_threshold=('p95_threshold', 'first'),
        p5_threshold=('p5_threshold', 'first')
    )
)

# Ensure integer counts
per_trace['count_high'] = per_trace['count_high'].astype(int)
per_trace['count_low'] = per_trace['count_low'].astype(int)

# Compute max_over_p95 safely
def safe_div(n, d):
    if pd.isna(n) or pd.isna(d) or d == 0:
        return np.nan
    return n / d

per_trace['max_over_p95'] = per_trace.apply(lambda r: safe_div(r['max_value_in_window'], r['p95_threshold']), axis=1)

# 2) Parse trace_name to extract direction and target component
def parse_trace_name(name, cmdb):
    # default
    direction = 'other'
    target = None
    if not isinstance(name, str):
        return direction, target
    if name.startswith('trace.'):
        parts = name.split('.')
        if len(parts) >= 2:
            core = parts[1]  # e.g., 'to_MG01' or 'from_IG02' or 'self'
            if core.startswith('to_'):
                direction = 'to'
                target = core.split('_',1)[1] if '_' in core else None
            elif core.startswith('from_'):
                direction = 'from'
                target = core.split('_',1)[1] if '_' in core else None
            elif core == 'self' or core.startswith('self'):
                direction = 'self'
                target = cmdb
            else:
                direction = core  # fallback to whatever
    return direction, target

parsed = per_trace.apply(lambda r: parse_trace_name(r['trace_name'], r['cmdb_id']), axis=1)
per_trace[['direction', 'target_component']] = pd.DataFrame(parsed.tolist(), index=per_trace.index)

# 3) Filter to edges: keep 'to' or 'from', and keep 'self' only if max_over_p95 > 1.5
edge_mask = (per_trace['direction'].isin(['to', 'from'])) | ((per_trace['direction'] == 'self') & (per_trace['max_over_p95'] > 1.5))
trace_edge_agg_df = per_trace[edge_mask].copy()

# Normalize earliest_high_ts to ISO strings with tz (or None)
def to_iso(ts):
    if pd.isna(ts):
        return None
    # ts should already be tz-aware (Asia/Shanghai) from trace_incident_df; if not, localize
    if ts.tzinfo is None:
        ts = ts.tz_localize(tz)
    else:
        ts = ts.tz_convert(tz)
    return ts.isoformat()

trace_edge_agg_df['earliest_high_ts_iso'] = trace_edge_agg_df['earliest_high_ts'].apply(to_iso)

# 4) Prepare compact top-20 sorted by max_over_p95 desc, tie-breaker count_high desc
top20_trace_edges = (
    trace_edge_agg_df
    .loc[:, ['cmdb_id', 'trace_name', 'direction', 'target_component', 'count_high', 'earliest_high_ts_iso', 'p95_threshold', 'max_value_in_window', 'max_over_p95']]
    .sort_values(by=['max_over_p95', 'count_high'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# Short list of distinct target_component values present in these top rows (limit to 20)
distinct_targets_top = pd.Series(top20_trace_edges['target_component'].dropna().unique()).astype(str).head(20)

# Keep full trace_edge_agg_df in kernel for next steps
trace_edge_agg_df.shape, top20_trace_edges, distinct_targets_top

```
Out[5]:
```
```python
summary_text = (
    "Trace-edge aggregation summary (incident window, times shown in UTC+8):\n\n"
    "- Total trace-edge series after filtering: 230 (stored in trace_edge_agg_df).\n\n"
    "- Top trace-edge series by max_over_p95 (top 10 of the top-20 sorted results):\n"
    "  1) dockerB2 | trace.self.duration_mean — max_over_p95=3.9968, count_high=2, earliest_high=2021-03-25T11:39:00+08:00, p95=0.084234, max=0.336667\n"
    "  2) MG02   | trace.from_dockerA1.duration_mean — max_over_p95=2.7148, count_high=2, earliest_high=2021-03-25T11:55:00+08:00\n"
    "  3) dockerA1 | trace.to_MG02.duration_mean — max_over_p95=2.7148, count_high=2, earliest_high=2021-03-25T11:55:00+08:00\n"
    "  4) MG01   | trace.from_dockerB1.duration_mean — max_over_p95=2.6302, count_high=2, earliest_high=2021-03-25T11:55:00+08:00\n"
    "  5) dockerB1 | trace.to_MG01.duration_mean — max_over_p95=2.6302, count_high=2, earliest_high=2021-03-25T11:55:00+08:00\n"
    "  6) MG01   | trace.to_dockerA1.duration_p95 — max_over_p95=2.4224, count_high=2, earliest_high=2021-03-25T11:40:00+08:00\n"
    "  7) dockerA1 | trace.from_MG01.duration_p95 — max_over_p95=2.4224, count_high=2, earliest_high=2021-03-25T11:40:00+08:00\n"
    "  8) dockerA1 | trace.self.duration_p95 — max_over_p95=1.9737, count_high=2, earliest_high=2021-03-25T11:40:00+08:00\n"
    "  9) MG01   | trace.to_dockerA1.duration_mean — max_over_p95=1.9084, count_high=2, earliest_high=2021-03-25T11:40:00+08:00\n"
    " 10) dockerA1 | trace.from_MG01.duration_mean — max_over_p95=1.9084, count_high=2, earliest_high=2021-03-25T11:40:00+08:00\n\n"
    "- Distinct target components among these top rows (short list):\n"
    "  [dockerB2, dockerA1, MG02, dockerB1, MG01, IG02, Tomcat01, dockerA2, IG01, Tomcat03]\n\n"
    "Key takeaways:\n"
    "- Several inter-component edges (from_/to_ relationships) show strong spikes versus their historical P95, notably interactions between docker nodes and MG components.\n"
    "- dockerB2 (self trace) has the largest relative spike (~4x P95). Multiple symmetric edges (e.g., dockerA1 <-> MG02, dockerB1 <-> MG01) appear with elevated ratios ~2.4–2.7.\n"
    "- Earliest high values for these top edges are mostly between 11:36 and 11:55, with many starting at or shortly after 11:40 or 11:55—indicating spikes during the incident window.\n"
    "- trace_edge_agg_df (230 rows), thresholds_trace_df, and trace_incident_df are stored in the kernel for further drill-down."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

((230, 14),      cmdb_id                         trace_name direction target_component  count_high       earliest_high_ts_iso  p95_threshold  max_value_in_window  max_over_p95
0   dockerB2           trace.self.duration_mean      self         dockerB2           2  2021-03-25T11:39:00+08:00       0.084234             0.336667      3.996830
1       MG02  trace.from_dockerA1.duration_mean      from         dockerA1           2  2021-03-25T11:55:00+08:00       0.138653             0.376422      2.714840
2   dockerA1        trace.to_MG02.duration_mean        to             MG02           2  2021-03-25T11:55:00+08:00       0.138653             0.376422      2.714840
3       MG01  trace.from_dockerB1.duration_mean      from         dockerB1           2  2021-03-25T11:55:00+08:00       0.130786             0.344000      2.630241
4   dockerB1        trace.to_MG01.duration_mean        to             MG01           2  2021-03-25T11:55:00+08:00       0.130786             0.344000      2.630241
5       MG01     trace.to_dockerA1.duration_p95        to         dockerA1           2  2021-03-25T11:40:00+08:00       1.044400             2.530000      2.422444
6   dockerA1       trace.from_MG01.duration_p95      from             MG01           2  2021-03-25T11:40:00+08:00       1.044400             2.530000      2.422444
7   dockerA1            trace.self.duration_p95      self         dockerA1           2  2021-03-25T11:40:00+08:00       0.009500             0.018750      1.973684
8       MG01    trace.to_dockerA1.duration_mean        to         dockerA1           2  2021-03-25T11:40:00+08:00       0.233843             0.446264      1.908392
9   dockerA1      trace.from_MG01.duration_mean      from             MG01           2  2021-03-25T11:40:00+08:00       0.233843             0.446264      1.908392
10      IG02            trace.self.duration_p95      self             IG02           2  2021-03-25T11:40:00+08:00       3.067650             5.787450      1.886607
11      IG02     trace.to_Tomcat01.duration_p95        to         Tomcat01           2  2021-03-25T11:46:00+08:00       3.118320             5.741000      1.841055
12  Tomcat01       trace.from_IG02.duration_p95      from             IG02           2  2021-03-25T11:46:00+08:00       3.118320             5.741000      1.841055
13      MG01  trace.from_dockerA1.duration_mean      from         dockerA1           2  2021-03-25T11:42:00+08:00       0.204454             0.370827      1.813743
14  dockerA1        trace.to_MG01.duration_mean        to             MG01           2  2021-03-25T11:42:00+08:00       0.204454             0.370827      1.813743
15      MG02  trace.from_dockerA2.duration_mean      from         dockerA2           2  2021-03-25T11:36:00+08:00       0.196646             0.352000      1.790020
16  dockerA2        trace.to_MG02.duration_mean        to             MG02           2  2021-03-25T11:36:00+08:00       0.196646             0.352000      1.790020
17  dockerA1           trace.self.duration_mean      self         dockerA1           2  2021-03-25T11:40:00+08:00       0.028640             0.051059      1.782786
18      IG01            trace.self.duration_p95      self             IG01           2  2021-03-25T11:39:00+08:00       3.115625             5.548000      1.780702
19      IG02     trace.to_Tomcat03.duration_p95        to         Tomcat03           2  2021-03-25T11:52:00+08:00       3.092125             5.457900      1.765097, 0    dockerB2
1    dockerA1
2        MG02
3    dockerB1
4        MG01
5        IG02
6    Tomcat01
7    dockerA2
8        IG01
9    Tomcat03
dtype: object)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel-stored variables:
# component_anomaly_df (metric-side component aggregation)
# component_trace_agg (trace-side component aggregation)
# trace_edge_agg_df, thresholds_trace_df, trace_incident_df, log_df, error_logs_df

# Defensive: ensure expected variables exist in the kernel namespace
# (If any is missing, this will raise NameError and should be fixed upstream)
_ = component_anomaly_df
_ = component_trace_agg
_ = trace_edge_agg_df
_ = thresholds_trace_df
_ = trace_incident_df
_ = log_df
_ = error_logs_df

# 1) Union set of components
metric_comps = set(component_anomaly_df['cmdb_id'].unique())
trace_comps = set(component_trace_agg['cmdb_id'].unique())
union_comps = sorted(metric_comps.union(trace_comps))

# 2) Prepare metric-side dataframe keyed by cmdb_id with required renamed columns
metric_side = component_anomaly_df[['cmdb_id', 'num_kpi_anomalies', 'total_anomaly_points', 'earliest_anomaly_ts', 'max_over_threshold_ratio', 'top_kpi']].copy()
metric_side = metric_side.rename(columns={
    'total_anomaly_points': 'metric_total_anomaly_points',
    'num_kpi_anomalies': 'num_kpi_anomalies',
    'earliest_anomaly_ts': 'metric_earliest_ts',
    'max_over_threshold_ratio': 'metric_max_ratio',
    'top_kpi': 'metric_top_kpi'
})

# Ensure metric_earliest_ts are ISO strings or None
def iso_or_none(x):
    if pd.isna(x):
        return None
    try:
        if isinstance(x, str):
            return x
        if x.tzinfo is None:
            return pd.Timestamp(x).tz_localize(tz).isoformat()
        return pd.Timestamp(x).tz_convert(tz).isoformat()
    except Exception:
        return None

metric_side['metric_earliest_ts'] = metric_side['metric_earliest_ts'].apply(iso_or_none)

# 3) Prepare trace-side dataframe keyed by cmdb_id
trace_side = component_trace_agg[['cmdb_id', 'num_trace_kpi_anomalies', 'total_trace_anomaly_points', 'earliest_trace_anomaly_ts', 'max_trace_over_threshold_ratio', 'top_trace_kpi']].copy()
trace_side = trace_side.rename(columns={
    'num_trace_kpi_anomalies': 'num_trace_kpi_anomalies',
    'total_trace_anomaly_points': 'total_trace_anomaly_points',
    'earliest_trace_anomaly_ts': 'earliest_trace_anomaly_ts',
    'max_trace_over_threshold_ratio': 'max_trace_over_threshold_ratio',
    'top_trace_kpi': 'top_trace_kpi'
})
# earliest_trace_anomaly_ts already ISO strings from prior steps; ensure None for NaN
trace_side['earliest_trace_anomaly_ts'] = trace_side['earliest_trace_anomaly_ts'].where(trace_side['earliest_trace_anomaly_ts'].notna(), None)

# 4) Log and error_log aggregation within incident window
start_ts = pd.Timestamp('2021-03-25 11:30:00').tz_localize(tz)
end_ts = pd.Timestamp('2021-03-25 12:00:00').tz_localize(tz)

# Filter log_df to incident window (log_df has 'ts' tz-aware)
log_incident = log_df[(log_df['ts'] >= start_ts) & (log_df['ts'] <= end_ts)].copy()

# log_rows_count per component
log_rows = log_incident.groupby('cmdb_id', as_index=False).size().rename(columns={'size': 'log_rows_count'})

# sum_log_error_count: sum of 'value' where log_name == 'log.error_count'
log_error = log_incident[log_incident['log_name'] == 'log.error_count'].groupby('cmdb_id', as_index=False)['value'].sum().rename(columns={'value': 'sum_log_error_count'})

# earliest_log_ts per component
earliest_log_ts = log_incident.groupby('cmdb_id', as_index=False)['ts'].min().rename(columns={'ts': 'earliest_log_ts'})
earliest_log_ts['earliest_log_ts'] = earliest_log_ts['earliest_log_ts'].apply(lambda x: x.tz_convert(tz).isoformat() if not pd.isna(x) else None)

# Merge log aggregates
log_agg = pd.merge(log_rows, log_error, on='cmdb_id', how='left')
log_agg = pd.merge(log_agg, earliest_log_ts, on='cmdb_id', how='left')
log_agg['sum_log_error_count'] = log_agg['sum_log_error_count'].fillna(0).astype(float)

# error_logs aggregation in incident window
# Ensure error_logs_df has 'ts' column; earlier we added 'ts' column
error_incident = error_logs_df[(error_logs_df['ts'] >= start_ts) & (error_logs_df['ts'] <= end_ts)].copy()
error_agg = error_incident.groupby('cmdb_id', as_index=False).size().rename(columns={'size': 'error_log_entries_count'})

# 5) Build combined_component_df by merging all sources on union set
combined_component_df = pd.DataFrame({'cmdb_id': union_comps})

# Merge metric side
combined_component_df = combined_component_df.merge(metric_side[['cmdb_id','metric_total_anomaly_points','num_kpi_anomalies','metric_earliest_ts','metric_max_ratio','metric_top_kpi']], on='cmdb_id', how='left')

# Merge trace side
combined_component_df = combined_component_df.merge(trace_side[['cmdb_id','total_trace_anomaly_points','num_trace_kpi_anomalies','earliest_trace_anomaly_ts','max_trace_over_threshold_ratio','top_trace_kpi']], on='cmdb_id', how='left')

# Merge log aggregates and error logs
combined_component_df = combined_component_df.merge(log_agg[['cmdb_id','log_rows_count','sum_log_error_count','earliest_log_ts']], on='cmdb_id', how='left')
combined_component_df = combined_component_df.merge(error_agg, on='cmdb_id', how='left')

# Fill missing numeric fields with zeros where appropriate
combined_component_df['metric_total_anomaly_points'] = combined_component_df['metric_total_anomaly_points'].fillna(0).astype(int)
combined_component_df['num_kpi_anomalies'] = combined_component_df['num_kpi_anomalies'].fillna(0).astype(int)
combined_component_df['metric_max_ratio'] = combined_component_df['metric_max_ratio'].where(combined_component_df['metric_max_ratio'].notna(), np.nan)
combined_component_df['total_trace_anomaly_points'] = combined_component_df['total_trace_anomaly_points'].fillna(0).astype(int)
combined_component_df['num_trace_kpi_anomalies'] = combined_component_df['num_trace_kpi_anomalies'].fillna(0).astype(int)
combined_component_df['max_trace_over_threshold_ratio'] = combined_component_df['max_trace_over_threshold_ratio'].where(combined_component_df['max_trace_over_threshold_ratio'].notna(), np.nan)

combined_component_df['log_rows_count'] = combined_component_df['log_rows_count'].fillna(0).astype(int)
combined_component_df['sum_log_error_count'] = combined_component_df['sum_log_error_count'].fillna(0).astype(float)
combined_component_df['error_log_entries_count'] = combined_component_df['error_log_entries_count'].fillna(0).astype(int)

# Normalize earliest timestamps to ISO strings (metric_earliest_ts may already be string)
combined_component_df['metric_earliest_ts'] = combined_component_df['metric_earliest_ts'].where(combined_component_df['metric_earliest_ts'].notna(), None)
combined_component_df['earliest_trace_anomaly_ts'] = combined_component_df['earliest_trace_anomaly_ts'].where(combined_component_df['earliest_trace_anomaly_ts'].notna(), None)

# 6) Produce compact table sorted by combined anomaly points desc
combined_component_df['combined_total_anomaly_points'] = combined_component_df['metric_total_anomaly_points'] + combined_component_df['total_trace_anomaly_points']

display_cols = [
    'cmdb_id',
    'metric_total_anomaly_points',
    'num_kpi_anomalies',
    'metric_earliest_ts',
    'metric_max_ratio',
    'metric_top_kpi',
    'total_trace_anomaly_points',
    'num_trace_kpi_anomalies',
    'earliest_trace_anomaly_ts',
    'max_trace_over_threshold_ratio',
    'top_trace_kpi',
    'log_rows_count',
    'sum_log_error_count',
    'error_log_entries_count'
]

top20_combined = combined_component_df.sort_values(['combined_total_anomaly_points','metric_max_ratio','max_trace_over_threshold_ratio'], ascending=[False, False, False])[display_cols].head(20).reset_index(drop=True)

# 7) Components anomalous in both metrics and traces
both_anomalous = combined_component_df[
    (combined_component_df['metric_total_anomaly_points'] > 0) &
    (combined_component_df['total_trace_anomaly_points'] > 0)
].copy()

# Indicate which of these have log errors or error log entries
both_anomalous['has_log_errors'] = both_anomalous['sum_log_error_count'] > 0
both_anomalous['has_error_logs'] = both_anomalous['error_log_entries_count'] > 0

both_anomalous_small = both_anomalous.loc[:, ['cmdb_id','metric_total_anomaly_points','total_trace_anomaly_points','sum_log_error_count','error_log_entries_count','has_log_errors','has_error_logs']].reset_index(drop=True)

# Store combined_component_df in kernel (variable created)
# Final compact outputs: top20_combined and both_anomalous_small (both are DataFrames)
top20_combined, both_anomalous_small, combined_component_df.shape

```
Out[6]:
```
Summary (plain English):

- Combined dataset: 29 components were combined from metric and trace aggregations (variable combined_component_df stored in kernel).

- Top affected components by combined anomaly points (metrics + traces):
  1. Mysql01 — metric-heavy: 2,652 metric anomaly points (226 KPIs); no trace anomalies; 42 log rows in window; no log.error or error_log entries.
  2. Mysql02 — 2,394 metric points (224 KPIs); no trace anomalies.
  3. Tomcat02 — 1,298 metric points (121 KPIs) and 69 trace anomaly points (18 trace KPIs); earliest trace anomaly 11:30; top trace KPI: trace.to_MG01.duration_p95.
  4. Tomcat04 — 1,264 metric points and 66 trace points; earliest trace anomaly 11:30.
  5. Tomcat03 — 1,255 metric points and 67 trace points.
  6. Tomcat01 — 1,151 metric points and 70 trace points.
  7. Redis02 / Redis01 — high metric anomalies (1,097 and 994 points) but no trace anomalies.
  8. MG02 / MG01 — both show substantial metric and trace anomalies (MG02: 790 metric + 164 trace; MG01: 747 metric + 166 trace).
  9. apache02 / apache01 — metric anomalies (~490 each); no trace anomalies; each has ~62 log rows in the window.
  10. docker nodes (dockerA1/A2, dockerB1/B2) show trace anomalies (many edges) but generally few or zero metric anomalies.

- Logs and error logs:
  - log.error_count sums in the incident window are 0 for components in the top results. Some components have log rows in the window (e.g., Mysql01 42, Tomcat nodes ~83–84, apache nodes 62), but no recorded log.error_count > 0 and error_logs.csv had no entries in the window.

- Components anomalous in both metrics and traces:
  - IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, Tomcat03, Tomcat04, dockerA2
  - None of these have recorded log.error_count or error_log entries in the incident window (so no direct error-log corroboration).

- Key takeaways and recommended next steps:
  - Many components (Mysql and Tomcat fleets, MG, Redis) show sustained metric anomalies (values >= historical P95) throughout the 31-minute incident window.
  - Several Tomcat, MG, and docker edges show trace latency spikes (trace KPIs exceeding P95), often starting at 11:30+08:00.
  - Because log.error_count and error_logs do not show errors for the components that are anomalous in both metrics and traces, investigate metrics and trace flows (top_kpi / top_trace_kpi per component) for root-cause (CPU, disk I/O, or inter-service latency), and drill into per-series time slices and traces for affected edges.
  - All relevant tables are stored in the kernel for follow-up: combined_component_df, component_anomaly_df, component_trace_agg, trace_edge_agg_df, thresholds_trace_df, metric_incident_df, trace_incident_df, log_incident (filtered), and error_incident.

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  metric_total_anomaly_points  num_kpi_anomalies         metric_earliest_ts  metric_max_ratio                                     metric_top_kpi  total_trace_anomaly_points  num_trace_kpi_anomalies  earliest_trace_anomaly_ts  max_trace_over_threshold_ratio                      top_trace_kpi  log_rows_count  sum_log_error_count  error_log_entries_count
0         Mysql01                         2652                226  2021-03-25T11:30:00+08:00          4.508701                         OSLinux-CPU_CPU_CPUCpuUtil                           0                        0                       None                             NaN                                NaN              42                  0.0                        0
1         Mysql02                         2394                224  2021-03-25T11:30:00+08:00          1.600000     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                           0                        0                       None                             NaN                                NaN               2                  0.0                        0
2        Tomcat02                         1298                121  2021-03-25T11:30:00+08:00          6.715785                             OSLinux-CPU_CPU_CPUWio                          69                       18  2021-03-25T11:30:00+08:00                        1.426897         trace.to_MG01.duration_p95              83                  0.0                        0
3        Tomcat04                         1264                121  2021-03-25T11:30:00+08:00          3.777228    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                          66                       18  2021-03-25T11:30:00+08:00                        1.377668        trace.to_MG02.duration_mean              84                  0.0                        0
4        Tomcat03                         1255                121  2021-03-25T11:30:00+08:00          4.000000                    OSLinux-CPU_CPU-2_SingleCpuidle                          67                       18  2021-03-25T11:30:00+08:00                        1.765097       trace.from_IG02.duration_p95              84                  0.0                        0
5        Tomcat01                         1151                118  2021-03-25T11:30:00+08:00          2.452456    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                          70                       18  2021-03-25T11:30:00+08:00                        1.841055       trace.from_IG02.duration_p95              83                  0.0                        0
6         Redis02                         1097                110  2021-03-25T11:30:00+08:00          5.429517                             OSLinux-CPU_CPU_CPUWio                           0                        0                       None                             NaN                                NaN               0                  0.0                        0
7         Redis01                          994                113  2021-03-25T11:30:00+08:00          3.469232                             OSLinux-CPU_CPU_CPUWio                           0                        0                       None                             NaN                                NaN               0                  0.0                        0
8            MG02                          790                104  2021-03-25T11:30:00+08:00          4.000000     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                         164                       42  2021-03-25T11:30:00+08:00                        2.714840  trace.from_dockerA1.duration_mean               0                  0.0                        0
9            MG01                          747                105  2021-03-25T11:30:00+08:00          2.197619                             OSLinux-CPU_CPU_CPUWio                         166                       42  2021-03-25T11:30:00+08:00                        2.630241  trace.from_dockerB1.duration_mean               0                  0.0                        0
10           IG01                          579                 88  2021-03-25T11:30:00+08:00          3.646505                             OSLinux-CPU_CPU_CPUWio                          60                       15  2021-03-25T11:32:00+08:00                        1.780702            trace.self.duration_p95               0                  0.0                        0
11           IG02                          549                 88  2021-03-25T11:30:00+08:00         10.000000       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                          67                       18  2021-03-25T11:30:00+08:00                        1.886607            trace.self.duration_p95               0                  0.0                        0
12       apache02                          492                 80  2021-03-25T11:30:00+08:00          3.127489                             OSLinux-CPU_CPU_CPUWio                           0                        0                       None                             NaN                                NaN              62                  0.0                        0
13       apache01                          491                 81  2021-03-25T11:30:00+08:00          1.778017                    OSLinux-CPU_CPU-0_SingleCpuUtil                           0                        0                       None                             NaN                                NaN              62                  0.0                        0
14       dockerA2                           44                 10  2021-03-25T11:30:00+08:00          1.110142  Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...                          72                       18  2021-03-25T11:30:00+08:00                        1.790020        trace.to_MG02.duration_mean               0                  0.0                        0
15       dockerA1                            0                  0                       None               NaN                                                NaN                          72                       18  2021-03-25T11:30:00+08:00                        2.714840        trace.to_MG02.duration_mean               0                  0.0                        0
16       dockerB1                            0                  0                       None               NaN                                                NaN                          69                       18  2021-03-25T11:30:00+08:00                        2.630241        trace.to_MG01.duration_mean               0                  0.0                        0
17       dockerB2                            0                  0                       None               NaN                                                NaN                          68                       18  2021-03-25T11:30:00+08:00                        3.996830           trace.self.duration_mean               0                  0.0                        0
18  ServiceTest10                           66                  4  2021-03-25T11:30:00+08:00          6.954892                                            app.mrt                           0                        0                       None                             NaN                                NaN               0                  0.0                        0
19   ServiceTest5                           66                  4  2021-03-25T11:30:00+08:00          3.554871                                            app.mrt                           0                        0                       None                             NaN                                NaN               0                  0.0                        0,     cmdb_id  metric_total_anomaly_points  total_trace_anomaly_points  sum_log_error_count  error_log_entries_count  has_log_errors  has_error_logs
0      IG01                          579                          60                  0.0                        0           False           False
1      IG02                          549                          67                  0.0                        0           False           False
2      MG01                          747                         166                  0.0                        0           False           False
3      MG02                          790                         164                  0.0                        0           False           False
4  Tomcat01                         1151                          70                  0.0                        0           False           False
5  Tomcat02                         1298                          69                  0.0                        0           False           False
6  Tomcat03                         1255                          67                  0.0                        0           False           False
7  Tomcat04                         1264                          66                  0.0                        0           False           False
8  dockerA2                           44                          72                  0.0                        0           False           False, (29, 16))```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables
_ = trace_edge_agg_df
_ = component_anomaly_df

# Identify metric-fault components from component_anomaly_df.
# Use the correct metric column name present in component_anomaly_df ('total_anomaly_points').
metric_fault_components = set(
    component_anomaly_df.loc[component_anomaly_df['total_anomaly_points'] > 0, 'cmdb_id'].unique()
)

# Filter trace edges whose target_component is in the metric-fault set
mask = trace_edge_agg_df['target_component'].notna() & trace_edge_agg_df['target_component'].isin(metric_fault_components)
trace_edges_to_metric_faults_df = trace_edge_agg_df[mask].copy()

# Normalize earliest-high timestamp (handle 'earliest_high_ts' or 'earliest_high_ts_iso')
def normalize_ts(x):
    if pd.isna(x):
        return pd.NaT
    ts = pd.to_datetime(x)
    if ts.tzinfo is None:
        ts = ts.tz_localize(tz)
    else:
        ts = ts.tz_convert(tz)
    return ts

if 'earliest_high_ts' in trace_edges_to_metric_faults_df.columns:
    trace_edges_to_metric_faults_df['earliest_high_ts_norm'] = trace_edges_to_metric_faults_df['earliest_high_ts'].apply(normalize_ts)
elif 'earliest_high_ts_iso' in trace_edges_to_metric_faults_df.columns:
    trace_edges_to_metric_faults_df['earliest_high_ts_norm'] = trace_edges_to_metric_faults_df['earliest_high_ts_iso'].apply(lambda x: normalize_ts(x) if pd.notna(x) else pd.NaT)
else:
    trace_edges_to_metric_faults_df['earliest_high_ts_norm'] = pd.NaT

# Ensure required columns exist to avoid KeyErrors
for col in ['count_high', 'p95_threshold', 'max_value_in_window', 'max_over_p95', 'direction', 'trace_name', 'target_component', 'cmdb_id']:
    if col not in trace_edges_to_metric_faults_df.columns:
        trace_edges_to_metric_faults_df[col] = pd.NA

# Sort by earliest_high_ts ascending (NaT last), then max_over_p95 descending
sorted_edges = trace_edges_to_metric_faults_df.sort_values(
    by=['earliest_high_ts_norm', 'max_over_p95'],
    ascending=[True, False],
    na_position='last'
).reset_index(drop=True)

# Format earliest_high_ts for display
sorted_edges['earliest_high_ts'] = sorted_edges['earliest_high_ts_norm'].apply(lambda x: x.isoformat() if not pd.isna(x) else None)

# Prepare top-20 compact table and rename source component
top20_edges = sorted_edges.loc[:, ['cmdb_id', 'trace_name', 'direction', 'target_component', 'earliest_high_ts', 'count_high', 'p95_threshold', 'max_value_in_window', 'max_over_p95']].head(20).rename(columns={'cmdb_id': 'source_component'})

# Short deduplicated list of target components in these top rows (limit 20)
distinct_targets = pd.Series(top20_edges['target_component'].dropna().unique()).head(20)

# Outputs stored in kernel: trace_edges_to_metric_faults_df (filtered full table), and show compact outputs
top20_edges, distinct_targets

```
Out[7]:
```
```python
# Compact plain-English summary based on trace_edges_to_metric_faults_df (stored in kernel)
tz = pytz.timezone('Asia/Shanghai')

# Reuse filtered DataFrame from kernel
_ = trace_edges_to_metric_faults_df

# Compute summary pieces
total_edges = int(trace_edges_to_metric_faults_df.shape[0])
# normalize earliest timestamps column name
if 'earliest_high_ts_norm' in trace_edges_to_metric_faults_df.columns:
    earliest_ts = trace_edges_to_metric_faults_df['earliest_high_ts_norm'].min()
else:
    earliest_ts = pd.to_datetime(trace_edges_to_metric_faults_df['earliest_high_ts']).min()
earliest_ts_iso = earliest_ts.tz_convert(tz).isoformat() if not pd.isna(earliest_ts) else None

# Top 5 earliest edges examples (source, trace_name, target, earliest, count_high, max_over_p95)
examples = (
    trace_edges_to_metric_faults_df
    .sort_values(['earliest_high_ts_norm','max_over_p95'], ascending=[True, False])
    .loc[:, ['cmdb_id','trace_name','target_component','earliest_high_ts','count_high','max_over_p95']]
    .head(5)
    .rename(columns={'cmdb_id':'source_component'})
    .to_dict(orient='records')
)

# Distinct target components in filtered rows (deduplicated)
distinct_targets = list(pd.Series(trace_edges_to_metric_faults_df['target_component'].dropna().unique()).astype(str))

summary_text = (
    f"Trace edges targeting components that have metric anomalies: {total_edges} edges found.\n"
    f"Earliest observed edge anomaly: {earliest_ts_iso} (Asia/Shanghai).\n\n"
    "Top 5 earliest edge examples:\n"
)

for ex in examples:
    summary_text += (
        f"- {ex['source_component']} -> {ex['target_component']} | {ex['trace_name']} | "
        f"earliest={ex['earliest_high_ts']} | count_high={int(ex['count_high'])} | max_over_p95={ex['max_over_p95']:.3f}\n"
    )

summary_text += "\nDistinct target components in these edges (deduplicated): " + ", ".join(distinct_targets) + ".\n\n"
summary_text += (
    "Notes: Many edges are between docker nodes and MG/Tomcat components (mutual from/to patterns). "
    "Some edges show notable relative spikes over P95 (e.g., ratios >1.5). The full filtered table is stored as "
    "'trace_edges_to_metric_faults_df' in the kernel for further drill-down."
)

# Display the plain-English summary string
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(   source_component                     trace_name direction target_component           earliest_high_ts  count_high  p95_threshold  max_value_in_window  max_over_p95
0              MG02    trace.to_dockerA2.row_count        to         dockerA2  2021-03-25T11:30:00+08:00           2     225.500000           267.000000      1.184035
1          dockerA2      trace.from_MG02.row_count      from             MG02  2021-03-25T11:30:00+08:00           2     225.500000           267.000000      1.184035
2          dockerB1  trace.from_MG01.duration_mean      from             MG01  2021-03-25T11:30:00+08:00           2       0.746829             0.840687      1.125676
3          dockerB2   trace.from_MG01.duration_p95      from             MG01  2021-03-25T11:30:00+08:00           2       4.953920             5.157500      1.041095
4              IG02    trace.to_Tomcat03.row_count        to         Tomcat03  2021-03-25T11:31:00+08:00           2     266.000000           277.000000      1.041353
5          Tomcat03      trace.from_IG02.row_count      from             IG02  2021-03-25T11:31:00+08:00           2     266.000000           277.000000      1.041353
6              MG02  trace.from_Tomcat01.row_count      from         Tomcat01  2021-03-25T11:31:00+08:00           2     254.000000           258.000000      1.015748
7          Tomcat01        trace.to_MG02.row_count        to             MG02  2021-03-25T11:31:00+08:00           2     254.000000           258.000000      1.015748
8          dockerB2      trace.from_MG02.row_count      from             MG02  2021-03-25T11:31:00+08:00           2     127.000000           129.000000      1.015748
9          dockerB1      trace.from_MG02.row_count      from             MG02  2021-03-25T11:32:00+08:00           2     357.500000           368.000000      1.029371
10         dockerB2      trace.from_MG01.row_count      from             MG01  2021-03-25T11:32:00+08:00           3     124.000000           126.000000      1.016129
11         dockerB1     trace.to_MG01.duration_p95        to             MG01  2021-03-25T11:32:00+08:00           3       1.015000             1.015000      1.000000
12         dockerA1        trace.to_MG02.row_count        to             MG02  2021-03-25T11:33:00+08:00           3      67.000000            92.000000      1.373134
13         dockerA1        trace.to_MG01.row_count        to             MG01  2021-03-25T11:33:00+08:00           2      70.500000            94.000000      1.333333
14             IG01    trace.to_Tomcat03.row_count        to         Tomcat03  2021-03-25T11:33:00+08:00           2     260.000000           272.000000      1.046154
15         Tomcat03      trace.from_IG01.row_count      from             IG01  2021-03-25T11:33:00+08:00           2     260.000000           272.000000      1.046154
16             MG01  trace.from_Tomcat02.row_count      from         Tomcat02  2021-03-25T11:33:00+08:00           2     247.000000           257.000000      1.040486
17         Tomcat02        trace.to_MG01.row_count        to             MG01  2021-03-25T11:33:00+08:00           2     247.000000           257.000000      1.040486
18             MG01  trace.from_dockerA2.row_count      from         dockerA2  2021-03-25T11:34:00+08:00           3      63.000000            96.000000      1.523810
19         dockerA2        trace.to_MG01.row_count        to             MG01  2021-03-25T11:34:00+08:00           3      63.000000            96.000000      1.523810, 0    dockerA2
1        MG02
2        MG01
3    Tomcat03
4        IG02
5    Tomcat01
6        IG01
7    Tomcat02
dtype: object)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables
_ = trace_edges_to_metric_faults_df
_ = combined_component_df

# Prepare source and target metric snippets from combined_component_df
source_metrics = combined_component_df[['cmdb_id','metric_total_anomaly_points','metric_earliest_ts','metric_max_ratio']].copy()
source_metrics = source_metrics.rename(columns={
    'cmdb_id': 'source_component',
    'metric_total_anomaly_points': 'source_metric_total_anomaly_points',
    'metric_earliest_ts': 'source_metric_earliest_ts',
    'metric_max_ratio': 'source_metric_max_ratio'
})

target_metrics = combined_component_df[['cmdb_id','metric_total_anomaly_points','metric_earliest_ts','metric_max_ratio']].copy()
target_metrics = target_metrics.rename(columns={
    'cmdb_id': 'target_component',
    'metric_total_anomaly_points': 'target_metric_total_anomaly_points',
    'metric_earliest_ts': 'target_metric_earliest_ts',
    'metric_max_ratio': 'target_metric_max_ratio'
})

# Start from trace_edges_to_metric_faults_df
edge_df = trace_edges_to_metric_faults_df.copy()

# Ensure we have a normalized earliest timestamp column for sorting
if 'earliest_high_ts_norm' not in edge_df.columns:
    def normalize_ts(x):
        if pd.isna(x):
            return pd.NaT
        ts = pd.to_datetime(x)
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        return ts.tz_convert(tz)
    if 'earliest_high_ts' in edge_df.columns:
        edge_df['earliest_high_ts_norm'] = edge_df['earliest_high_ts'].apply(normalize_ts)
    elif 'earliest_high_ts_iso' in edge_df.columns:
        edge_df['earliest_high_ts_norm'] = edge_df['earliest_high_ts_iso'].apply(lambda x: normalize_ts(x) if pd.notna(x) else pd.NaT)
    else:
        edge_df['earliest_high_ts_norm'] = pd.NaT

# Merge source metrics (based on edge source cmdb_id -> combined_component_df.cmdb_id)
edge_df = edge_df.merge(source_metrics, left_on='cmdb_id', right_on='source_component', how='left')

# Merge target metrics (based on target_component -> combined_component_df.cmdb_id)
edge_df = edge_df.merge(target_metrics, on='target_component', how='left')

# Fill missing numeric totals with 0
edge_df['source_metric_total_anomaly_points'] = edge_df['source_metric_total_anomaly_points'].fillna(0).astype(int)
edge_df['target_metric_total_anomaly_points'] = edge_df['target_metric_total_anomaly_points'].fillna(0).astype(int)

# Normalize earliest metric ts strings to None if missing (they may already be ISO strings)
edge_df['source_metric_earliest_ts'] = edge_df['source_metric_earliest_ts'].where(edge_df['source_metric_earliest_ts'].notna(), None)
edge_df['target_metric_earliest_ts'] = edge_df['target_metric_earliest_ts'].where(edge_df['target_metric_earliest_ts'].notna(), None)

# Keep final selected columns and rename for output
edge_source_target_metric_join_df = edge_df.loc[:, [
    'cmdb_id',                     # source component
    'target_component',
    'trace_name',
    'earliest_high_ts',            # human-readable iso (from earlier steps)
    'earliest_high_ts_norm',       # used for sorting but not shown
    'count_high',
    'max_over_p95',
    'source_metric_total_anomaly_points',
    'source_metric_earliest_ts',
    'source_metric_max_ratio',
    'target_metric_total_anomaly_points',
    'target_metric_earliest_ts',
    'target_metric_max_ratio'
]].copy()

# Rename for clarity
edge_source_target_metric_join_df = edge_source_target_metric_join_df.rename(columns={
    'cmdb_id': 'source_component',
    'earliest_high_ts': 'earliest_high_ts_trace'
})

# Sort by earliest_high_ts asc (NaT last), then by target_metric_max_ratio desc
edge_source_target_metric_join_df['earliest_sort'] = edge_source_target_metric_join_df['earliest_high_ts_norm']
edge_source_target_metric_join_df = edge_source_target_metric_join_df.sort_values(
    by=['earliest_sort', 'target_metric_max_ratio'],
    ascending=[True, False],
    na_position='last'
).reset_index(drop=True)

# Prepare compact top-20 view
edge_top20 = edge_source_target_metric_join_df.loc[:, [
    'source_component','target_component','trace_name','earliest_high_ts_trace','count_high','max_over_p95',
    'source_metric_total_anomaly_points','source_metric_earliest_ts','source_metric_max_ratio',
    'target_metric_total_anomaly_points','target_metric_earliest_ts','target_metric_max_ratio'
]].head(20).copy()

# Identify edges where target has metric anomalies but source does not
candidate_mask = (edge_source_target_metric_join_df['target_metric_total_anomaly_points'] > 0) & (edge_source_target_metric_join_df['source_metric_total_anomaly_points'] == 0)
candidate_edges = edge_source_target_metric_join_df.loc[candidate_mask, ['source_component','target_component','trace_name','earliest_high_ts_trace','count_high','max_over_p95','target_metric_total_anomaly_points']].drop_duplicates().reset_index(drop=True)

# Deduplicated small list of candidate edges (source -> target) (limit to 50 for compactness)
candidate_edges_dedup = candidate_edges.loc[:, ['source_component','target_component']].drop_duplicates().head(50)

# Store full result in kernel variable (edge_source_target_metric_join_df already created)
edge_source_target_metric_join_df.shape, edge_top20, candidate_edges_dedup

```
Out[8]:
```
Summary of edge-level join (caller ↔ callee metric-fault info):

- Dataset: edge_source_target_metric_join_df contains 158 joined edges (trace-edge rows joined with component metric summaries). Top 20 rows were produced, sorted by earliest trace high timestamp (ascending) and then by target metric max ratio (descending).

- Earliest examples (top rows):
  1. dockerA2 -> MG02 | trace.from_MG02.row_count | earliest 2021-03-25T11:30:00+08:00 | count_high=2 | max_over_p95≈1.18  
     - source metric: 44 anomaly points (metric_earliest_ts 11:30, max_ratio≈1.11)  
     - target metric: 790 anomaly points (metric_earliest_ts 11:30, max_ratio=4.00)
  2. dockerB1 -> MG01 | trace.from_MG01.duration_mean | earliest 11:30 | count_high=2 | max_over_p95≈1.13  
     - source metric: 0 metric anomaly points  
     - target metric: 747 metric anomaly points (max_ratio≈2.20)
  3. dockerB2 -> MG01 | trace.from_MG01.duration_p95 | earliest 11:30 | count_high=2 | max_over_p95≈1.04  
     - source metric: 0 ; target metric: 747 (max_ratio≈2.20)
  4. MG02 -> dockerA2 | trace.to_dockerA2.row_count | earliest 11:30 | ... (mirrors row 1 with source/target swapped)
  5. Tomcat03 -> IG02 | trace.from_IG02.row_count | earliest 11:31 | target IG02 has 549 metric anomaly points (max_ratio=10.0)

- Key observation: many early edges (around 11:30–11:34) connect docker nodes, MG, Tomcat, and IG components. In several edges the callee (target) has large metric anomaly counts while the caller (source) has few or no metric anomalies.

- Strong candidate edges where the callee (target) shows metric anomalies but the caller (source) does not:
  - dockerB1 -> MG01
  - dockerB2 -> MG01
  - dockerB2 -> MG02
  - dockerB1 -> MG02
  - dockerA1 -> MG02
  - dockerA1 -> MG01
  (These are deduplicated candidate edges from the join and are good leads indicating the target may be the root cause.)

- Interpretation / next steps:
  - Edges where target_metric_total_anomaly_points > 0 but source_metric_total_anomaly_points == 0 are prioritized: they suggest the callee (target) is experiencing metric anomalies while callers are not, making the target a strong candidate root cause.
  - For each candidate edge, drill down into the target component’s top metric KPIs (metric_top_kpi) and the trace details (trace_name, timestamps) to confirm causality and investigate resource KPIs (CPU, disk I/O, memory).
  - Use the stored table edge_source_target_metric_join_df in the kernel for detailed inspection and to trace specific timestamps and full series.

All results are stored in the kernel (edge_source_target_metric_join_df and earlier tables) for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

((158, 14),    source_component target_component                     trace_name    earliest_high_ts_trace  count_high  max_over_p95  source_metric_total_anomaly_points  source_metric_earliest_ts  source_metric_max_ratio  target_metric_total_anomaly_points  target_metric_earliest_ts  target_metric_max_ratio
0          dockerA2             MG02      trace.from_MG02.row_count 2021-03-25 11:30:00+08:00           2      1.184035                                  44  2021-03-25T11:30:00+08:00                 1.110142                                 790  2021-03-25T11:30:00+08:00                 4.000000
1          dockerB1             MG01  trace.from_MG01.duration_mean 2021-03-25 11:30:00+08:00           2      1.125676                                   0                       None                      NaN                                 747  2021-03-25T11:30:00+08:00                 2.197619
2          dockerB2             MG01   trace.from_MG01.duration_p95 2021-03-25 11:30:00+08:00           2      1.041095                                   0                       None                      NaN                                 747  2021-03-25T11:30:00+08:00                 2.197619
3              MG02         dockerA2    trace.to_dockerA2.row_count 2021-03-25 11:30:00+08:00           2      1.184035                                 790  2021-03-25T11:30:00+08:00                 4.000000                                  44  2021-03-25T11:30:00+08:00                 1.110142
4          Tomcat03             IG02      trace.from_IG02.row_count 2021-03-25 11:31:00+08:00           2      1.041353                                1255  2021-03-25T11:30:00+08:00                 4.000000                                 549  2021-03-25T11:30:00+08:00                10.000000
5              IG02         Tomcat03    trace.to_Tomcat03.row_count 2021-03-25 11:31:00+08:00           2      1.041353                                 549  2021-03-25T11:30:00+08:00                10.000000                                1255  2021-03-25T11:30:00+08:00                 4.000000
6          Tomcat01             MG02        trace.to_MG02.row_count 2021-03-25 11:31:00+08:00           2      1.015748                                1151  2021-03-25T11:30:00+08:00                 2.452456                                 790  2021-03-25T11:30:00+08:00                 4.000000
7          dockerB2             MG02      trace.from_MG02.row_count 2021-03-25 11:31:00+08:00           2      1.015748                                   0                       None                      NaN                                 790  2021-03-25T11:30:00+08:00                 4.000000
8              MG02         Tomcat01  trace.from_Tomcat01.row_count 2021-03-25 11:31:00+08:00           2      1.015748                                 790  2021-03-25T11:30:00+08:00                 4.000000                                1151  2021-03-25T11:30:00+08:00                 2.452456
9          dockerB1             MG02      trace.from_MG02.row_count 2021-03-25 11:32:00+08:00           2      1.029371                                   0                       None                      NaN                                 790  2021-03-25T11:30:00+08:00                 4.000000
10         dockerB1             MG01     trace.to_MG01.duration_p95 2021-03-25 11:32:00+08:00           3      1.000000                                   0                       None                      NaN                                 747  2021-03-25T11:30:00+08:00                 2.197619
11         dockerB2             MG01      trace.from_MG01.row_count 2021-03-25 11:32:00+08:00           3      1.016129                                   0                       None                      NaN                                 747  2021-03-25T11:30:00+08:00                 2.197619
12             MG01         Tomcat02  trace.from_Tomcat02.row_count 2021-03-25 11:33:00+08:00           2      1.040486                                 747  2021-03-25T11:30:00+08:00                 2.197619                                1298  2021-03-25T11:30:00+08:00                 6.715785
13             IG01         Tomcat03    trace.to_Tomcat03.row_count 2021-03-25 11:33:00+08:00           2      1.046154                                 579  2021-03-25T11:30:00+08:00                 3.646505                                1255  2021-03-25T11:30:00+08:00                 4.000000
14         dockerA1             MG02        trace.to_MG02.row_count 2021-03-25 11:33:00+08:00           3      1.373134                                   0                       None                      NaN                                 790  2021-03-25T11:30:00+08:00                 4.000000
15         Tomcat03             IG01      trace.from_IG01.row_count 2021-03-25 11:33:00+08:00           2      1.046154                                1255  2021-03-25T11:30:00+08:00                 4.000000                                 579  2021-03-25T11:30:00+08:00                 3.646505
16         Tomcat02             MG01        trace.to_MG01.row_count 2021-03-25 11:33:00+08:00           2      1.040486                                1298  2021-03-25T11:30:00+08:00                 6.715785                                 747  2021-03-25T11:30:00+08:00                 2.197619
17         dockerA1             MG01        trace.to_MG01.row_count 2021-03-25 11:33:00+08:00           2      1.333333                                   0                       None                      NaN                                 747  2021-03-25T11:30:00+08:00                 2.197619
18         Tomcat01             IG02      trace.from_IG02.row_count 2021-03-25 11:34:00+08:00           2      1.041181                                1151  2021-03-25T11:30:00+08:00                 2.452456                                 549  2021-03-25T11:30:00+08:00                10.000000
19         Tomcat04             IG02      trace.from_IG02.row_count 2021-03-25 11:34:00+08:00           2      1.174853                                1264  2021-03-25T11:30:00+08:00                 3.777228                                 549  2021-03-25T11:30:00+08:00                10.000000,   source_component target_component
0         dockerB1             MG01
1         dockerB2             MG01
2         dockerB2             MG02
3         dockerB1             MG02
6         dockerA1             MG02
7         dockerA1             MG01)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel-stored tables
_ = combined_component_df
_ = edge_source_target_metric_join_df
_ = trace_edges_to_metric_faults_df

# 1) Candidate set: components with metric_total_anomaly_points > 0
candidates = combined_component_df[combined_component_df['metric_total_anomaly_points'] > 0].copy()
candidates = candidates[['cmdb_id', 'metric_total_anomaly_points', 'metric_max_ratio', 'total_trace_anomaly_points', 'metric_earliest_ts']].copy()
candidates = candidates.rename(columns={'metric_earliest_ts': 'earliest_metric_ts'})

# 2) Compute inbound edge aggregates from trace_edges_to_metric_faults_df
# Ensure normalized earliest timestamp column exists
if 'earliest_high_ts_norm' in trace_edges_to_metric_faults_df.columns:
    ts_col = 'earliest_high_ts_norm'
elif 'earliest_high_ts' in trace_edges_to_metric_faults_df.columns:
    # normalize to tz-aware
    trace_edges_to_metric_faults_df['earliest_high_ts_norm'] = pd.to_datetime(trace_edges_to_metric_faults_df['earliest_high_ts']).apply(
        lambda x: x.tz_localize(tz) if x.tzinfo is None else x.tz_convert(tz)
    )
    ts_col = 'earliest_high_ts_norm'
else:
    trace_edges_to_metric_faults_df['earliest_high_ts_norm'] = pd.NaT
    ts_col = 'earliest_high_ts_norm'

inbound_grp = trace_edges_to_metric_faults_df.groupby('target_component', as_index=False).agg(
    inbound_edges_count=('trace_name', 'size'),
    earliest_inbound_ts=(ts_col, 'min')
)

# 2b) inbound_edges_from_healthy_callers from edge_source_target_metric_join_df
# Ensure numeric column for source metric totals exists
if 'source_metric_total_anomaly_points' not in edge_source_target_metric_join_df.columns:
    edge_source_target_metric_join_df['source_metric_total_anomaly_points'] = 0

healthy_callers = (
    edge_source_target_metric_join_df[edge_source_target_metric_join_df['source_metric_total_anomaly_points'] == 0]
    .groupby('target_component', as_index=False)
    .agg(inbound_edges_from_healthy_callers=('source_component', 'nunique'))
)

# 3) Merge inbound aggregates into candidates
prior = candidates.merge(inbound_grp, left_on='cmdb_id', right_on='target_component', how='left')
prior = prior.merge(healthy_callers, left_on='cmdb_id', right_on='target_component', how='left', suffixes=('', '_hc'))

# Clean up and fill defaults
prior['inbound_edges_count'] = prior['inbound_edges_count'].fillna(0).astype(int)
# earliest_inbound_ts to ISO string or None
def ts_to_iso(x):
    if pd.isna(x):
        return None
    try:
        # ensure tz-aware Asia/Shanghai
        if x.tzinfo is None:
            x = x.tz_localize(tz)
        else:
            x = x.tz_convert(tz)
        return x.isoformat()
    except Exception:
        return None

prior['earliest_inbound_trace_edge_ts'] = prior['earliest_inbound_ts'].apply(ts_to_iso)
prior['inbound_edges_from_healthy_callers'] = prior['inbound_edges_from_healthy_callers'].fillna(0).astype(int)

# Keep required columns and ensure numeric types
prior = prior[[
    'cmdb_id',
    'metric_total_anomaly_points',
    'metric_max_ratio',
    'total_trace_anomaly_points',
    'earliest_metric_ts',
    'earliest_inbound_trace_edge_ts',
    'inbound_edges_count',
    'inbound_edges_from_healthy_callers'
]].copy()

# 4) Compute priority_score (treat NaN metric_max_ratio as 0 for scoring)
prior['metric_max_ratio_for_score'] = prior['metric_max_ratio'].fillna(0).astype(float)
prior['priority_score'] = (
    prior['metric_max_ratio_for_score'] * 100.0
    + prior['metric_total_anomaly_points'] / 10.0
    + prior['total_trace_anomaly_points'] * 5.0
    + prior['inbound_edges_from_healthy_callers'] * 200.0
)

# 5) Sort and select top 10
prioritized_candidates_df = prior.sort_values('priority_score', ascending=False).reset_index(drop=True)

top10_candidates = prioritized_candidates_df.loc[:, [
    'cmdb_id',
    'metric_total_anomaly_points',
    'metric_max_ratio',
    'total_trace_anomaly_points',
    'inbound_edges_count',
    'inbound_edges_from_healthy_callers',
    'earliest_metric_ts',
    'earliest_inbound_trace_edge_ts',
    'priority_score'
]].head(10).copy()

# 6) Top single candidate row
top_candidate_row = top10_candidates.head(1).copy()
# Extract explicit earliest times for the top candidate
top_candidate_times = top_candidate_row.loc[:, ['cmdb_id','earliest_metric_ts','earliest_inbound_trace_edge_ts']]

# Store result in kernel
prioritized_candidates_df = prioritized_candidates_df  # stored

# Display compact outputs: top10_candidates, top_candidate_row (times)
top10_candidates, top_candidate_times

```
Out[9]:
```
```python
summary_text = (
    "Prioritized metric-fault candidate summary (top 10):\n\n"
    "Method: candidates are components with metric anomalies; score = metric_max_ratio*100 + metric_points/10 + trace_points*5 + healthy-inbound-edges*200.\n\n"
    "Top candidates:\n"
    "1) MG02 — score 1899.0\n"
    "   - metric_total_anomaly_points: 790\n"
    "   - metric_max_ratio: 4.00\n"
    "   - total_trace_anomaly_points: 164\n"
    "   - inbound_edges_count: 36\n"
    "   - inbound_edges_from_healthy_callers: 3\n"
    "   - earliest_metric_ts: 2021-03-25T11:30:00+08:00\n"
    "   - earliest_inbound_trace_edge_ts: 2021-03-25T11:30:00+08:00\n\n"
    "2) MG01 — score 1724.46\n"
    "   - metric_total_anomaly_points: 747, metric_max_ratio: 2.20, trace_points: 166, inbound_edges: 36, healthy_inbound: 3\n"
    "   - earliest_metric_ts: 2021-03-25T11:30:00+08:00\n\n"
    "3) IG02 — score 1389.90\n"
    "   - metric_points: 549, metric_max_ratio: 10.00 (very high), trace_points: 67, inbound_edges: 13\n"
    "   - earliest_metric_ts: 2021-03-25T11:30:00+08:00; earliest inbound trace ~11:31\n\n"
    "4) Tomcat02 — score 1146.38 (metric-heavy: 1298 points, metric_max_ratio 6.72)\n"
    "5) Tomcat03 — score 860.50\n"
    "6) Tomcat04 — score 834.12\n"
    "7) IG01 — score 722.55\n"
    "8) Mysql01 — score 716.07 (very large metric points but few trace anomalies)\n"
    "9) Tomcat01 — score 710.35\n"
    "10) ServiceTest10 — score 702.09\n\n"
    "Key interpretation:\n"
    "- MG02 is the top candidate (highest score). It has large metric anomalies, many inbound trace anomalies, and 3 inbound edges from callers that show no metric faults (healthy callers). This pattern—high metric anomalies on the callee plus healthy callers—makes MG02 a strong root-cause candidate.\n"
    "- MG01 is the second-highest candidate with a similar pattern (3 healthy inbound callers).\n"
    "- IG02 has a very high metric_max_ratio (10x), making it notable even with fewer total trace points.\n"
    "- Mysql01 has very large metric anomaly volume but minimal trace anomalies; it’s important but may be a local resource issue rather than an inter-service propagation issue.\n\n"
    "Recommended next steps:\n"
    "1) Drill into MG02 and MG01 top metric KPIs (metric_top_kpi) around 2021-03-25T11:30:00+08:00 to identify resource impacts (CPU, disk I/O, memory).\n"
    "2) Inspect inbound trace edges flagged from healthy callers to confirm directional impact and correlate trace latencies with metric spikes.\n"
    "3) Use the stored table 'prioritized_candidates_df' and edge-level tables in the kernel for targeted forensic queries.\n"
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id  metric_total_anomaly_points  metric_max_ratio  total_trace_anomaly_points  inbound_edges_count  inbound_edges_from_healthy_callers         earliest_metric_ts earliest_inbound_trace_edge_ts  priority_score
0           MG02                          790          4.000000                         164                   36                                   3  2021-03-25T11:30:00+08:00      2021-03-25T11:30:00+08:00     1899.000000
1           MG01                          747          2.197619                         166                   36                                   3  2021-03-25T11:30:00+08:00      2021-03-25T11:30:00+08:00     1724.461852
2           IG02                          549         10.000000                          67                   13                                   0  2021-03-25T11:30:00+08:00      2021-03-25T11:31:00+08:00     1389.900000
3       Tomcat02                         1298          6.715785                          69                   12                                   0  2021-03-25T11:30:00+08:00      2021-03-25T11:33:00+08:00     1146.378464
4       Tomcat03                         1255          4.000000                          67                   12                                   0  2021-03-25T11:30:00+08:00      2021-03-25T11:31:00+08:00      860.500000
5       Tomcat04                         1264          3.777228                          66                   12                                   0  2021-03-25T11:30:00+08:00      2021-03-25T11:34:00+08:00      834.122772
6           IG01                          579          3.646505                          60                   13                                   0  2021-03-25T11:30:00+08:00      2021-03-25T11:33:00+08:00      722.550487
7        Mysql01                         2652          4.508701                           0                    0                                   0  2021-03-25T11:30:00+08:00                           None      716.070114
8       Tomcat01                         1151          2.452456                          70                   12                                   0  2021-03-25T11:30:00+08:00      2021-03-25T11:31:00+08:00      710.345642
9  ServiceTest10                           66          6.954892                           0                    0                                   0  2021-03-25T11:30:00+08:00                           None      702.089250,   cmdb_id         earliest_metric_ts earliest_inbound_trace_edge_ts
0    MG02  2021-03-25T11:30:00+08:00      2021-03-25T11:30:00+08:00)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel-stored variables
_ = thresholds_df
_ = metric_incident_df
_ = edge_source_target_metric_join_df
_ = trace_edges_to_metric_faults_df
_ = combined_component_df

# 1) MG02 top metric KPI faults
cm = 'MG02'

# Merge metric incident rows for MG02 with global thresholds (thresholds_df must be precomputed)
mg_metric = metric_incident_df[metric_incident_df['cmdb_id'] == cm].copy()
mg_metric = mg_metric.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')

# Flag anomaly rows where value >= p95_threshold (only when p95 exists)
mg_metric['is_anomaly'] = mg_metric['p95_threshold'].notna() & (mg_metric['value'] >= mg_metric['p95_threshold'])

# Aggregate per kpi_name
agg = (
    mg_metric
    .groupby('kpi_name', as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_ts=('ts', lambda s: s[mg_metric.loc[s.index, 'is_anomaly']].min() if s.notna().any() else pd.NaT),
        p95_threshold=('p95_threshold', 'first'),
        max_value_in_window=('value', 'max')
    )
)

# Safe ratio
def safe_ratio(maxv, p95):
    if pd.isna(p95) or p95 == 0:
        return np.nan
    return float(maxv) / float(p95)

agg['max_over_p95'] = agg.apply(lambda r: safe_ratio(r['max_value_in_window'], r['p95_threshold']), axis=1)

# Resource tagging from kpi_name keywords (case-insensitive)
def tag_resource(k):
    if not isinstance(k, str):
        return 'unknown'
    l = k.lower()
    if 'jvm' in l:
        return 'JVM'
    if any(tok in l for tok in ['cpu', 'cpuid', 'cpuload']):
        return 'CPU'
    if any(tok in l for tok in ['mem', 'memory', 'heap', 'noheap']):
        return 'Memory'
    if any(tok in l for tok in ['dsk', 'disk', 'localdisk', 'read', 'write']):
        return 'Disk I/O'
    if any(tok in l for tok in ['network', 'net', 'rx', 'tx', 'tcp']):
        return 'Network'
    return 'other'

agg['resource_type'] = agg['kpi_name'].apply(tag_resource)

# Format earliest_anomaly_ts as ISO strings in Asia/Shanghai or None
def to_iso(ts):
    if pd.isna(ts):
        return None
    if ts.tzinfo is None:
        ts = ts.tz_localize(tz)
    else:
        ts = ts.tz_convert(tz)
    return ts.isoformat()

agg['earliest_anomaly_ts'] = agg['earliest_anomaly_ts'].apply(to_iso)

# Select and sort by max_over_p95 desc, top 20
mg02_kpi_faults_df = agg.loc[:, ['kpi_name','resource_type','anomaly_count','earliest_anomaly_ts','p95_threshold','max_value_in_window','max_over_p95']].sort_values(by=['max_over_p95'], ascending=False).reset_index(drop=True)
mg02_kpi_faults_df = mg02_kpi_faults_df.head(20)

# 2) MG02 inbound healthy-caller trace edges
# edge_source_target_metric_join_df has source_component, target_component, earliest_high_ts_trace, count_high, max_over_p95, source_metric_total_anomaly_points, target_metric_total_anomaly_points
edges = edge_source_target_metric_join_df.copy()

# Ensure earliest timestamp normalized for sorting
if 'earliest_high_ts_trace' in edges.columns:
    edges['earliest_high_ts_norm'] = pd.to_datetime(edges['earliest_high_ts_trace']).apply(lambda x: x.tz_localize(tz) if x.tzinfo is None else x.tz_convert(tz))
else:
    # fallback if different column exists
    if 'earliest_high_ts' in edges.columns:
        edges['earliest_high_ts_norm'] = pd.to_datetime(edges['earliest_high_ts']).apply(lambda x: x.tz_localize(tz) if x.tzinfo is None else x.tz_convert(tz))
    else:
        edges['earliest_high_ts_norm'] = pd.NaT

mg02_inbound_from_healthy_df = (
    edges[
        (edges['target_component'] == cm) &
        (edges['source_metric_total_anomaly_points'] == 0)
    ]
    .loc[:, ['source_component','trace_name','earliest_high_ts_trace','count_high','max_over_p95','source_metric_total_anomaly_points','target_metric_total_anomaly_points']]
    .sort_values(by=['earliest_high_ts_trace', 'max_over_p95'], ascending=[True, False])
    .head(20)
    .reset_index(drop=True)
)

# Store results in kernel variables (already assigned) and display compact outputs
mg02_kpi_faults_df, mg02_inbound_from_healthy_df

```
Out[10]:
```
```python
# Plain-English summary for MG02 based on stored kernel tables (mg02_kpi_faults_df and mg02_inbound_from_healthy_df)
summary_text = (
    "MG02 quick findings (timestamps in Asia/Shanghai):\n\n"
    "1) Metric KPI faults (top signals):\n"
    "   - Disk I/O stands out as the dominant issue: e.g., sda_DSKBps max_over_p95=4.0 (earliest 11:56), "
    "sda_DSKRead max_over_p95≈3.91 (11:42), DSKRTps ≈3.40 (11:42), and sda/sdb write/read ratios >1.2–3.9.  \n"
    "   - CPU-related indicators also elevated (CPU WIO, CPUSysTime, CPULoad) with max_over_p95 ≈1.07–2.38 (earliest around 11:30–11:43).  \n"
    "   - JVM memory shows small rise (heap used ratio ≈1.02) but far smaller than disk/CPU spikes.\n\n"
    "2) Timing: metric anomalies for MG02 appear as early as ~11:30 (CPU sys/load), with pronounced disk spikes around 11:42 and 11:56.\n\n"
    "3) Trace context — inbound edges from seemingly healthy callers:\n"
    "   - Several callers that have no metric faults call MG02 and show trace spikes (examples): dockerB2, dockerB1, dockerA1.  \n"
    "   - Earliest inbound trace highs to MG02 start ~11:31; notable trace ratios include dockerA1 -> MG02 (trace.to_MG02.duration_mean) up to 2.71 at 11:55, "
    "and dockerB2 -> MG02 duration_mean ≈1.73 at 11:39. Many inbound edges have count_high>1 during the window.\n\n"
    "4) Interpretation and recommendation:\n"
    "   - MG02 is a high-priority candidate root cause. The strongest signals point to disk I/O saturation (sda/sdb) combined with CPU WIO. "
    "Trace evidence shows callers are often healthy while MG02 is experiencing issues, supporting MG02 as the likely callee-side fault.  \n"
    "   - Next steps: drill into MG02 disk metrics (per-device I/O, queue length), inspect system I/O wait and processes causing I/O, "
    "correlate the exact trace spans at the listed timestamps (11:30–11:56), and review MG02 logs around those times.\n\n"
    "Stored objects in kernel for follow-up: mg02_kpi_faults_df (metric KPIs) and mg02_inbound_from_healthy_df (inbound healthy-call edges)."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name resource_type  anomaly_count        earliest_anomaly_ts  p95_threshold  max_value_in_window  max_over_p95
0      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps      Disk I/O              1  2021-03-25T11:56:00+08:00   2.500000e-01         1.000000e+00      4.000000
1     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead      Disk I/O              1  2021-03-25T11:42:00+08:00   2.546667e+02         9.954667e+02      3.908901
2     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps      Disk I/O              1  2021-03-25T11:42:00+08:00   1.837500e+00         6.250000e+00      3.401360
3                              OSLinux-CPU_CPU_CPUWio           CPU              2  2021-03-25T11:43:00+08:00   1.189878e+00         2.837765e+00      2.384920
4    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite      Disk I/O              1  2021-03-25T11:42:00+08:00   2.290002e+01         5.120000e+01      2.235805
5   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...      Disk I/O              1  2021-03-25T11:56:00+08:00   4.250000e-01         8.000000e-01      1.882353
6   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...      Disk I/O              1  2021-03-25T11:42:00+08:00   2.250000e-01         3.000000e-01      1.333333
7    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite      Disk I/O              1  2021-03-25T11:40:00+08:00   1.604467e+03         2.006267e+03      1.250426
8     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps      Disk I/O              1  2021-03-25T11:40:00+08:00   2.291675e+00         2.766700e+00      1.207283
9            OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem        Memory              1  2021-03-25T11:43:00+08:00   1.413500e+02         1.660000e+02      1.174390
10                         OSLinux-CPU_CPU_CPUSysTime           CPU              2  2021-03-25T11:30:00+08:00   1.114800e+00         1.268508e+00      1.137879
11  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...      Disk I/O              1  2021-03-25T11:42:00+08:00   1.764800e+03         2.006000e+03      1.136673
12    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil       Network              1  2021-03-25T11:44:00+08:00   7.870000e-03         8.500000e-03      1.080051
13                            OSLinux-CPU_CPU_CPULoad           CPU              2  2021-03-25T11:31:00+08:00   1.500000e+00         1.610000e+00      1.073333
14    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum       Network              2  2021-03-25T11:40:00+08:00   4.664000e+03         4.873000e+03      1.044811
15    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec       Network              1  2021-03-25T11:36:00+08:00   7.974498e+05         8.224960e+05      1.031408
16                    OSLinux-CPU_CPU-0_SingleCpuUtil           CPU              1  2021-03-25T11:55:00+08:00   6.685920e+00         6.882200e+00      1.029357
17          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed           JVM              2  2021-03-25T11:43:00+08:00   9.606379e+08         9.794033e+08      1.019534
18                         OSLinux-CPU_CPU_CPUCpuUtil           CPU              1  2021-03-25T11:43:00+08:00   2.914744e+01         2.963840e+01      1.016844
19                    OSLinux-CPU_CPU-1_SingleCpuUtil           CPU              1  2021-03-25T11:47:00+08:00   6.060380e+00         6.131300e+00      1.011702,    source_component                     trace_name    earliest_high_ts_trace  count_high  max_over_p95  source_metric_total_anomaly_points  target_metric_total_anomaly_points
0          dockerB2      trace.from_MG02.row_count 2021-03-25 11:31:00+08:00           2      1.015748                                   0                                 790
1          dockerB1      trace.from_MG02.row_count 2021-03-25 11:32:00+08:00           2      1.029371                                   0                                 790
2          dockerA1        trace.to_MG02.row_count 2021-03-25 11:33:00+08:00           3      1.373134                                   0                                 790
3          dockerA1      trace.from_MG02.row_count 2021-03-25 11:35:00+08:00           2      1.140312                                   0                                 790
4          dockerB2    trace.to_MG02.duration_mean 2021-03-25 11:39:00+08:00           2      1.728852                                   0                                 790
5          dockerB1   trace.from_MG02.duration_p95 2021-03-25 11:39:00+08:00           2      1.113084                                   0                                 790
6          dockerB2   trace.from_MG02.duration_p95 2021-03-25 11:39:00+08:00           2      1.111920                                   0                                 790
7          dockerB2  trace.from_MG02.duration_mean 2021-03-25 11:39:00+08:00           2      1.059349                                   0                                 790
8          dockerB2     trace.to_MG02.duration_p95 2021-03-25 11:39:00+08:00           3      1.001233                                   0                                 790
9          dockerB1  trace.from_MG02.duration_mean 2021-03-25 11:40:00+08:00           2      1.718197                                   0                                 790
10         dockerA1   trace.from_MG02.duration_p95 2021-03-25 11:40:00+08:00           2      1.483575                                   0                                 790
11         dockerB1        trace.to_MG02.row_count 2021-03-25 11:40:00+08:00           2      1.348226                                   0                                 790
12         dockerA1  trace.from_MG02.duration_mean 2021-03-25 11:40:00+08:00           2      1.315299                                   0                                 790
13         dockerB2        trace.to_MG02.row_count 2021-03-25 11:40:00+08:00           2      1.236121                                   0                                 790
14         dockerB1     trace.to_MG02.duration_p95 2021-03-25 11:41:00+08:00           2      1.003051                                   0                                 790
15         dockerA1     trace.to_MG02.duration_p95 2021-03-25 11:46:00+08:00           2      1.519508                                   0                                 790
16         dockerA1    trace.to_MG02.duration_mean 2021-03-25 11:55:00+08:00           2      2.714840                                   0                                 790
17         dockerB1    trace.to_MG02.duration_mean 2021-03-25 11:55:00+08:00           2      1.524329                                   0                                 790)```
```